# Analysis code for final benchmarks

Python imports used (maybe not all are required for this notebook):
```
jupyter
pandas
matplotlib
altair
vega_datasets
altair_transform
altair_data_server
vegafusion[embed]
```

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import altair as alt
import json

import altair as alt
import vegafusion as vf
vf.enable(
    #mimetype="html", # use for interactive plots and for saving plots
    mimetype="png",  # use "png" to make plots show offline in notebook (but they will be blurry for reasons I don't understand)
    embed_options={"scaleFactor": 4},  # scaleFactor 4 makes exported png look good
    row_limit=150000,  # allows you to include plots with more datapoints
)



## Data paths

In [ ]:
artifact_path = Path("data/artifact/")
artifact_data_path = artifact_path / "benchmarks"

dataset_stats_path = artifact_path / "dataset-statistics"
lemma_dist_data_path = dataset_stats_path / "lemma_distance.txt"
lemma_size_data_path = dataset_stats_path / "lemma_size.txt"
lemma_new_tactics_data_path = dataset_stats_path / "lemmas_new_tactics.txt"
lemma_no_new_tactics_data_path = dataset_stats_path / "lemmas_no_new_tactics.txt"
lemmas_ssr_data_path = dataset_stats_path / "ssrlemmas.txt"
lemmas_normal_data_path = dataset_stats_path / "normallemmas.txt"

In [ ]:
assert artifact_path.exists()
assert artifact_data_path.exists()

assert dataset_stats_path.exists()
assert lemma_dist_data_path.exists()
assert lemma_size_data_path.exists()
assert lemma_new_tactics_data_path.exists()
assert lemma_no_new_tactics_data_path.exists()
assert lemmas_ssr_data_path.exists()
assert lemmas_normal_data_path.exists()


## Get the benchmark data

In [ ]:
paths = [
    ("CoqHammer CVC4", "CoqHammer-CVC4"),
    ("CoqHammer Eprover", "CoqHammer-Eprover"),
    ("CoqHammer Vampire", "CoqHammer-Vampire"),
    ("CoqHammer Z3", "CoqHammer-Z3"),
    ("CoqHammer 'best' tactic", "CoqHammer-best"),
    ("CoqHammer 'sauto' tactic", "CoqHammer-sauto"),
    ("GNN No names Update no definitions", "G2T-Anon-Frozen"),
    ("GNN No names Update all definitions", "G2T-Anon-Recalc"),
    ("GNN No names Update new definitions", "G2T-Anon-Update"),
    ("GNN Names Update no definitions", "G2T-Named-Frozen"),
    ("GNN Names Update all definitions", "G2T-Named-Recalc"),
    ("GNN Names Update new definitions", "G2T-Named-Update"),
    ("GNN No Definitions Update no definitions", "G2T-NoDef-Frozen"),
    ("LSHF", "LSHF"),
    ("LSHF extreme tactic decomposition", "LSHF-Extreme-Tactic-Decomposition"),
    ("Transformer CPU Big", "Transformer-CPU-Big"),
    ("Transformer CPU Small", "Transformer-CPU-Small"),
    ("Transformer GPU", "Transformer-GPU"),
    ("firstorder eauto with *", "firstorder-auto"),
    ("k-NN", "k-NN"),
]

In [ ]:
benchmark_paths2 = [
    {"label": label, "path": f"{artifact_data_path}/{path}/Set-Tactician-Benchmark-{seconds}./combined.bench", "version": "new"}
    for label, path in paths
    for seconds in [300, 600, 900]
]
benchmark_paths2

In [ ]:
def get_df(path: Path):
  df = pd.read_csv(path, sep="\t", names=["package", "theorem", "time_limit", "path", "found_proof", "time", "steps"], dtype=str)
  df["solved"] = df["path"].notna()
  return df

def get_df2(path: Path):
  df = pd.read_csv(path, sep="\t", names=["package", "theorem", "time_limit", "time", "steps", "messages", "path", "found_proof"], dtype=str)
  df["solved"] = df["path"].notna()
  df["error"] = df["time"].isna()
  return df

dfs = []
for d in benchmark_paths2:
  print(d)
  path = Path(d["path"])
  if not path.exists():
    continue
  if "version" in d and d["version"] == "new":
    df = get_df2(path)
  else:
    df = get_df(path)
  df["run"] = d["label"]
  dfs.append(df)

results_df = pd.concat(dfs)

# fill in omitted theorems
results_df["omitted"] = False
results_df["package_theorem"] = results_df["package"] + "_" + results_df["theorem"]
results_df["package_theorem"] = pd.Categorical(results_df["package_theorem"], categories=results_df["package_theorem"].unique())
thms = results_df.groupby("package_theorem")["theorem"].first()
pkgs = results_df.groupby("package_theorem")["package"].first()
results_df = results_df.groupby(["run", "package_theorem"], as_index=False).first()
results_df["package_theorem"] = results_df["package_theorem"].astype("str")
results_df["omitted"] = results_df["omitted"].fillna(True)
results_df["error"] = results_df["error"].fillna(True)  # includes omitted rows
results_df["solved"] = results_df["solved"].fillna(False)  # includes omitted rows
results_df["theorem"] = results_df["theorem"].fillna(results_df["package_theorem"].map(thms))
results_df["package"] = results_df["package"].fillna(results_df["package_theorem"].map(pkgs))

results_df["time_limit"] = results_df["time_limit"].astype("float")
results_df["time"] = results_df["time"].astype("float")

thm_lists = {}
for k, df in results_df[results_df["run"] == "LSHF"].groupby("time_limit"):
  thm_lists[k] = list(df["theorem"])

assert len(thm_lists[300.0]) == 4321
assert len(thm_lists[600.0]) == 2000

results_df["short_list"] = results_df["theorem"].isin(thm_lists[600.0])
results_df["long_list"] = results_df["theorem"].isin(thm_lists[300.0] + thm_lists[600.0])

results_2000_df = results_df[results_df["short_list"]].copy()
results_2000_df["solved"] = results_2000_df["solved"] & (results_2000_df["time"] <= 600.0)

results_500_each_df = results_df[results_df["long_list"]].copy()
results_500_each_df["solved"] = results_500_each_df["solved"] & (results_500_each_df["time"] <= 300.0)

del results_df

In [ ]:
results_2000_df

In [ ]:
results_500_each_df

## Make artificial runs with combos

In [ ]:
def get_combo(results_time_df, combo, time_limit):
    mask = np.zeros(len(results_time_df), dtype=bool)
    for run, fraction in combo.items():
        mask = mask | (results_time_df[run] <= time_limit * fraction)
    
    results_time_df = results_time_df[mask].copy()

    scales = {run: 1/np.array(fraction) for run, fraction in combo.items()}
    for run, scale in scales.items():
        results_time_df[run] = results_time_df[run] * scale
    
    results_time_df = results_time_df[list(combo.keys())].reset_index()
    results_time_df = pd.melt(results_time_df, id_vars=["theorem"], var_name='run', value_name='time')
    results_time_df = results_time_df.sort_values("time")
    results_time_df = results_time_df.groupby("theorem").first()
    return results_time_df

def get_combo_size(results_time_df, combo, time_limit):
    mask = np.zeros(len(results_time_df), dtype=bool)
    for run, fraction in combo.items():
        mask = mask | (results_time_df[run] <= time_limit * fraction)
    return mask.sum()

def get_combo_time(results_time_df, combo, time_limit):
    df = get_combo(results_time_df, combo, time_limit)
    return np.minimum(df["time"], 600.0).sum()

def make_combo_results_df(results_df, combo, run_name, time_limit):
    results_time_df = results_df.copy()
    results_time_df["time"] = np.where(results_time_df["solved"], results_time_df["time"], np.inf)
    results_time_df = results_time_df.pivot_table(index="theorem", columns="run", values="time")
    results_time_df = results_time_df.fillna(np.inf)

    df = results_df.copy()
    df = df[df["run"].isin(combo.keys())]
    df = df.drop(columns="time")
    combined_df = get_combo(results_time_df, combo, time_limit)
    combined_df = combined_df.reset_index().set_index(["theorem", "run"])
    combined_df = combined_df.join(df.set_index(["theorem", "run"]))
    # add back in all unsolved theorems
    # just use data from first available run for each theorem
    df = df.groupby("package_theorem").first().reset_index()
    df = df.set_index(["package_theorem", "theorem", "package"])[[]]
    combined_df = df.join(combined_df.reset_index().set_index(["package_theorem", "theorem", "package"])).reset_index()
    combined_df["run"] = run_name
    combined_df["solved"] = combined_df["solved"].fillna(False)
    combined_df["error"] = combined_df["error"].fillna(False)
    combined_df["omitted"] = combined_df["omitted"].fillna(False)
    combined_df["steps"] = combined_df["steps"].fillna(0)
    combined_df["messages"] = combined_df["messages"].fillna(0)
    return combined_df
    
    

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer")]
coq_hammers = df["run"].unique()
combo = [run for run in coq_hammers if "sauto" not in run]
combo = {run: 1.0/len(combo) for run in combo}
print(combo)
combined_hammer_2000_df = make_combo_results_df(results_2000_df, combo, run_name="CoqHammer combined", time_limit=10.0*60.0)
combined_hammer_2000_df

In [ ]:
df = results_500_each_df.copy()
df = df[df["run"].str.contains("CoqHammer")]
coq_hammers = df["run"].unique()
combo = [run for run in coq_hammers if "sauto" not in run]
combo = {run: 1.0/len(combo) for run in combo}
print(combo)
combined_hammer_500_each_df = make_combo_results_df(results_500_each_df, combo, run_name="CoqHammer combined", time_limit=5*60.0)
combined_hammer_500_each_df

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer")]
combo = [run for run in coq_hammers if "sauto" not in run]
combo = {run: 1.0 for run in combo}
print(combo)
combined_hammer_no_rescale_2000_df = make_combo_results_df(
    results_2000_df,
    combo,
    run_name="CoqHammer combined (unscaled)",
    time_limit=10*60.0
)
combined_hammer_no_rescale_2000_df

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer")]
coq_hammers = df["run"].unique()
combo = [run for run in coq_hammers if "Vampire" in run or "best" in run]
combo = {run: 1.0/len(combo) for run in combo}
print(combo)
coq_hammer_vampire_best_2000_df = make_combo_results_df(results_2000_df, combo, run_name="CoqHammer best+vampire", time_limit=10*60.0)
coq_hammer_vampire_best_2000_df

In [ ]:
two_best_models_combined_2000_df = make_combo_results_df(
    results_2000_df, 
    combo= {"k-NN": 1.0/2, "GNN No names Update new definitions": 1.0/2},
    run_name="Top 2 combined",
    time_limit=10*60.0
)
two_best_models_combined_2000_df

In [ ]:
two_best_models_combined_500_each_df = make_combo_results_df(
    results_500_each_df, 
    combo= {"k-NN": 1.0/2, "GNN No names Update new definitions": 1.0/2},
    run_name="Top 2 combined",
    time_limit=5*60.0
)
combined_hammer_500_each_df = make_combo_results_df(results_500_each_df, combo, run_name="CoqHammer combined", time_limit=5*60.0)
two_best_models_combined_500_each_df

In [ ]:
#combo = {"k-NN": 10.0*60.0/3, "GNN No names Update new definitions": 10.0*60.0/3}
#combo.update({ch: t/3 for ch, t in coq_hammers_combo.items()})
#three_best_models_combined_2000_df = make_combo_results_df(
#    results_2000_df, 
#    combo=combo,
#    run_name="Top 3 combined"
#)
#three_best_models_combined_2000_df

In [ ]:
combo = {'GNN No names Update no definitions': 1.0/4, 'k-NN': 1.0/4, 'Transformer GPU': 1.0/4, "CoqHammer 'best' tactic": 1.0/4}
four_best_models_combined_2000_df = make_combo_results_df(
    results_2000_df, 
    combo=combo,
    run_name="Top 4 combined",
    time_limit=10*60.0
)
four_best_models_combined_2000_df


In [ ]:
combo = {run: 10*60.0/len(results_2000_df["run"].unique()) for run in results_2000_df["run"].unique()}
all_combined_2000_df = make_combo_results_df(
    results_2000_df, 
    combo=combo,
    run_name="All combined",
    time_limit=10*60.0
)
all_combined_2000_df

In [ ]:
knn_g2t_named_update_combined_2000_df = make_combo_results_df(
    results_2000_df, 
    combo= {"k-NN": 1.0/2, "GNN Names Update new definitions": 1.0/2},
    run_name="knn + G2T-Named-Update",
    time_limit=10*60.0
)
knn_g2t_named_update_combined_2000_df

In [ ]:
knn_g2t_nodef_frozen_combined_2000_df = make_combo_results_df(
    results_2000_df, 
    combo= {"k-NN": 1.0/2, "GNN No Definitions Update no definitions": 1.0/2},
    run_name="knn + G2T-NoDef-Frozen",
    time_limit=10*60.0
)
knn_g2t_nodef_frozen_combined_2000_df

In [ ]:
knn_transformer_combined_2000_df = make_combo_results_df(
    results_2000_df, 
    combo= {"k-NN": 1.0/2, "Transformer GPU": 1.0/2},
    run_name="knn + Transformer-GPU",
    time_limit=10*60.0
)
knn_transformer_combined_2000_df

In [ ]:
knn_coqhammer_combined_2000_df = make_combo_results_df(
    pd.concat([results_2000_df, combined_hammer_2000_df]), 
    combo= {"k-NN": 1.0/2, "CoqHammer combined": 1.0/2},
    run_name="knn + CoqHammer",
    time_limit=10*60.0
)
knn_coqhammer_combined_2000_df

In [ ]:
results_combined_2000_df = pd.concat([
    results_2000_df,
    combined_hammer_2000_df,
    combined_hammer_no_rescale_2000_df,
    coq_hammer_vampire_best_2000_df,
    two_best_models_combined_2000_df,
    #three_best_models_combined_2000_df,
    four_best_models_combined_2000_df,
    all_combined_2000_df,
    knn_coqhammer_combined_2000_df,
    knn_g2t_named_update_combined_2000_df,
    knn_g2t_nodef_frozen_combined_2000_df,
    knn_transformer_combined_2000_df,
]).reset_index()
results_combined_2000_df

In [ ]:
results_combined_500_each_df = pd.concat([
    results_500_each_df,
    combined_hammer_500_each_df,
    two_best_models_combined_500_each_df,
]).reset_index()
results_combined_500_each_df

## Simple statistics for the data

In [ ]:
results_2000_df.groupby("run")[["error", "omitted"]].agg(["sum"])

In [ ]:
results_500_each_df.groupby("run")["theorem"].agg(["count", "nunique"])

In [ ]:
# error rate
results_2000_df.groupby("run")["error"].agg(["sum", "mean"])

In [ ]:
# error rate
results_500_each_df.groupby("run")["error"].agg(["sum", "mean"])

In [ ]:
# error rate
df = results_500_each_df.groupby(["run", "package"])["error"].agg(["sum", "mean"])
df = df[df["mean"] > .05]
df

In [ ]:
results_500_each_df[results_500_each_df["run"].str.contains("k-NN")].groupby("package").size().sort_values()

In [ ]:
results_500_each_df.groupby(["run", "package"]).size()

In [ ]:
results_2000_df.groupby("run")["solved"].agg(["count", "sum", "mean"])

In [ ]:
df = results_2000_df.copy()
df["end_early"] = (results_2000_df["time"] < .95 * 600.0) & ~results_2000_df["solved"]
df.groupby("run")["end_early"].agg(["count", "sum", "mean"])

In [ ]:
df = results_500_each_df.copy()
df.pivot_table(values="solved", columns="run", index="package", aggfunc="sum")

In [ ]:
df = results_500_each_df.copy()
print(df.pivot_table(values="solved", columns="run", index="package", aggfunc="sum").to_latex())

In [ ]:
df = results_500_each_df.copy()
df.pivot_table(values="solved", columns="run", index="package", aggfunc="sum")[["GNN Names Update new definitions", "GNN No names Update new definitions", "k-NN"]]

In [ ]:
df = results_500_each_df.copy()
df = df.pivot_table(values="solved", columns="run", index="package", aggfunc="mean")
df

In [ ]:
df = results_500_each_df.copy()
print(df.pivot_table(values="solved", columns="run", index="package", aggfunc="mean").to_latex())

In [ ]:
x = """package total thms2000 thms500each sum
coq-bbv.1.3 653 59 441 500
coq-bits.1.1.0 428 35 393 428
coq-bytestring.0.9.0 19 1 18 19
coq-ceres.0.4.0 103 12 91 103
coq-corn.8.16.0 6757 584 0 584
coq-gaia-stern.1.15 923 62 438 500
coq-haskell.1.0.0 174 21 153 174
coq-hott.8.11 3883 342 158 500
coq-iris-heap-lang.3.4.0 331 38 293 331
coq-mathcomp-apery.1.0.1 512 38 462 500
coq-mathcomp-odd-order.1.14.0 1602 129 371 500
coq-poltac.0.8.11 309 20 289 309
coq-printf.2.0.0 21 1 20 21
coq-qcert.2.2.0 4565 341 159 500
coq-smtcoq.2.0+8.11 999 90 410 500
coq-tlc.20200328 2191 195 305 500
coq-topology.10.0.1 352 32 320 352"""
headers = x.split("\n")[0].split(" ")
data = [y.split(" ") for y in x.split("\n")[1:]]
package_sizes = pd.DataFrame(data, columns=headers)
package_sizes["total"] = package_sizes["total"].astype("int")
package_sizes

## Run statistics for paper

In [ ]:
# table of speeds

df = results_2000_df.copy()
df = df[~df["run"].str.contains("CoqHammer")]
df = df[~df["run"].str.contains("firstorder")]
df = df[df["messages"].notna()]
df["messages"] = df["messages"].astype(int)
df = df[df["steps"].notna()]
df["steps"] = df["steps"].astype(int)
df2 = df.groupby("run")[["messages", "steps", "time"]].sum()
df2["tactics_executed_per_second"] = df2["steps"] / df2["time"]
df2["model_calls_per_second"] = df2["messages"] / df2["time"]
df2 = df2[["model_calls_per_second", "tactics_executed_per_second"]].sort_values("model_calls_per_second", ascending=False)
df2

In [ ]:
# table of speeds

df = results_2000_df.copy()
df = df[~df["run"].str.contains("CoqHammer")]
df = df[~df["run"].str.contains("firstorder")]
df = df[df["messages"].notna()]
df["messages"] = df["messages"].astype(int)
df = df[df["steps"].notna()]
df["steps"] = df["steps"].astype(int)
df["tactics_executed_per_second"] = df["steps"] / df["time"]
df["model_calls_per_second"] = df["messages"] / df["time"]
df2 = df.groupby("run")[["model_calls_per_second", "tactics_executed_per_second"]].mean()
df2["model_calls_per_second_std"] = df.groupby("run")["model_calls_per_second"].std()
df2["tactics_executed_per_second_std"] = df.groupby("run")["tactics_executed_per_second"].std()
df2 = df2.sort_values("model_calls_per_second", ascending=False)
df2

In [ ]:
# table of speeds

df = results_2000_df.copy()
df = df[~df["run"].str.contains("CoqHammer")]
df = df[~df["run"].str.contains("firstorder")]
df = df[df["messages"].notna()]
df["messages"] = df["messages"].astype(int)
df = df[df["steps"].notna()]
df["steps"] = df["steps"].astype(int)
df["tactics_executed_per_second"] = df["steps"] / df["time"]
df["model_calls_per_second"] = df["messages"] / df["time"]
df2 = df.groupby("run")[["model_calls_per_second", "tactics_executed_per_second"]].mean()
df2["model_calls_per_second_std"] = df.groupby("run")["model_calls_per_second"].std()
df2["tactics_executed_per_second_std"] = df.groupby("run")["tactics_executed_per_second"].std()
df2 = df2.sort_values("model_calls_per_second", ascending=False)
df2

In [ ]:
# # table of speeds using only positive information

df = results_2000_df.copy()
df = df[~df["run"].str.contains("CoqHammer")]
df = df[~df["run"].str.contains("firstorder")]
df = df[df["messages"].notna()]
df = df[df["solved"]]
df["messages"] = df["messages"].astype(int)
df = df[df["steps"].notna()]
df["steps"] = df["steps"].astype(int)
df2 = df.groupby("run")[["messages", "steps", "time"]].sum()
df2["tactics_executed_per_second"] = df2["steps"] / df2["time"]
df2["model_calls_per_second"] = df2["messages"] / df2["time"]
df2 = df2[["model_calls_per_second", "tactics_executed_per_second"]].sort_values("model_calls_per_second", ascending=False)
df2

## Helper functions to aggregate data by time, model_calls or executed_tactics

In [ ]:
def aggregate_by_time_step(
    df: pd.DataFrame,
    time_column: str,
    groupby: list,
    packages_to_exclude: list = [],
    theorems_to_exclude: list = [],
    up_to_percentile: float = 1.0,
    max_time: float = None,
) -> pd.DataFrame:
    df = df.copy()
    
    # exclude packages
    df = df[~df["package"].isin(packages_to_exclude)]
    # exclude theorems
    df = df[~df["theorem"].isin(theorems_to_exclude)]

    
    # count total theorems in each run
    df["run_total"] = df.groupby(groupby)["solved"].transform("count") 
    df["pct_solved"] = df["solved"] / df["run_total"]
    # now that we know how many total rows there are we can excluded all errors since they don't have time measurements
    df = df[~df["error"]]

    # process three possible time indices by ordering them by time and counting their place
    df["seconds"] = df["time"]
    df["tactics_executed"] = df["steps"].astype("int")
    df["model_calls"] = df["messages"].astype("int")

    df["seconds_ix"] = df.sort_values(["seconds"]).groupby(groupby)["seconds"].cumcount()
    df["model_calls_ix"] = df.sort_values(["model_calls"]).groupby(groupby)["model_calls"].cumcount()
    df["tactics_executed_ix"] = df.sort_values(["tactics_executed"]).groupby(groupby)["tactics_executed"].cumcount()
    
    # now that we have a cum count, we can restrict to only those solved
    df = df[df["solved"]]

    # sort by time_column and calculate cumulative statistics
    time_df = df.copy().sort_values([time_column])
    time_df["cum_solved"] = time_df.groupby(groupby)["solved"].cumsum()
    time_df["cum_pct_solved"] = time_df["cum_solved"] / time_df["run_total"]
    
    # calculate cumulative times for all three time columns, but only one is ordered
    for tm_column in ["seconds", "tactics_executed", "model_calls"]:
        time_df[f"cum_{tm_column}"] = time_df.groupby(groupby)[tm_column].cumsum()    
    # compare to other times
    for tm_column in ["seconds", "tactics_executed", "model_calls"]:
        if tm_column == time_column:
            continue
        time_df[f"{tm_column}_per_{time_column}"] = time_df[f"cum_{tm_column}"] / time_df[f"cum_{time_column}"]
    # remove data after a particular percentile
    time_df = time_df[time_df[f"{time_column}_ix"] <= time_df["run_total"] * up_to_percentile]
    
    return time_df

def add_final_times(data, groupby, monotone_cols, time_col, max_time):
    data = data[groupby + monotone_cols + [time_col]]
    final = data.groupby(groupby).last().reset_index()
    final[time_col] = max_time
    return pd.concat([data, final])

## Main plots

In [ ]:
time_df = aggregate_by_time_step(
    results_combined_2000_df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
)

selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q"),
    y=alt.Y("cum_pct_solved:Q"),
    color="run:N",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "cum_solved:Q", "cum_pct_solved:Q", "seconds:Q"]
).add_params(
    selection
)

In [ ]:
time_df = aggregate_by_time_step(
    results_combined_2000_df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
)

selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log")),
    y=alt.Y("cum_pct_solved:Q"),
    color="run:N",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "cum_solved:Q", "cum_pct_solved:Q", "seconds:Q"]
).add_params(
    selection
)

In [ ]:
time_df = aggregate_by_time_step(
    results_2000_df,
    time_column="tactics_executed",
    groupby=["run"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
    up_to_percentile=1.0,
)

selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(time_df).mark_line().encode(
    x=alt.X("tactics_executed:Q", scale=alt.Scale(type="log")),
    y="cum_pct_solved:Q",
    color="run",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "cum_solved:Q", "cum_pct_solved:Q", "seconds:Q"]
).add_params(
    selection
)

In [ ]:
time_df = aggregate_by_time_step(
    results_2000_df,
    time_column="model_calls",
    groupby=["run"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
    up_to_percentile=0.5,
)

selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(time_df).mark_line().encode(
    x=alt.X("model_calls:Q", scale=alt.Scale(type="log")),
    y="cum_pct_solved:Q",
    color="run",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "cum_solved:Q", "cum_pct_solved:Q", "seconds:Q"]
).add_params(
    selection
)

In [ ]:
time_df = aggregate_by_time_step(
    results_2000_df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
)

selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(time_df).mark_line().encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log")),
    y=alt.Y("tactics_executed_per_seconds:Q", scale=alt.Scale(type="log")),
    color="run",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "cum_solved:Q", "cum_pct_solved:Q", "tactics_executed_per_second:Q"]
).add_params(
    selection
)

In [ ]:
time_df = aggregate_by_time_step(
    results_2000_df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
)

selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(time_df).mark_line().encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log")),
    y=alt.Y("model_calls_per_seconds:Q", scale=alt.Scale(type="log")),
    color="run",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "cum_solved:Q", "cum_pct_solved:Q", "model_calls_per_seconds:Q"]
).add_params(
    selection
)

## Plots for paper

In [ ]:
paper_models_table = pd.DataFrame([
    {"run": "Top 2 combined", "model": "G2T-Anon-Update + k-NN", "venn_color": "lightgrey", "color": "#9d755d", "steps": False},
    {"run": "k-NN", "model": "k-NN", "venn_color": "#7E00FF", "color": "#b279a2", "steps": True},
    {"run": "GNN No names Update new definitions", "model": "G2T-Anon-Update", "venn_color": "#FF5400", "color": "#e45756", "steps": True},
    {"run": "GNN Names Update new definitions", "model": "G2T-Named-Update", "venn_color": "lightgrey", "color": "#4c78a8", "steps": True},
    {"run": "GNN No Definitions Update no definitions", "model": "G2T-NoDef-Frozen", "venn_color": "lightgrey", "color": "#f58518", "steps": True},
    {"run": "CoqHammer combined", "model": "CoqHammer combined", "venn_color": "#00A61F", "color": "#54a24b", "steps": False},
    {"run": "Transformer GPU", "model": "Transformer-GPU", "venn_color": "#F7FF00", "color": "#eeca3b", "steps": True},
    {"run": "Transformer CPU Big", "model": "Transformer-CPU", "venn_color": "lightgrey", "color": "#ff9da6", "steps": True},
    {"run": "firstorder eauto with *", "model": "firstorder", "venn_color": "lightgrey", "color": "#72b7b2", "steps": False},
])

#df = df[df["run"].isin([
#    "k-NN",
#    "GNN No names Update new definitions",
#    "GNN Names Update new definitions",
#    "GNN No Definitions Update no definitions",
#    "Transformer GPU",
#    "Transformer CPU Big",
#    "Top 2 combined",
#    "CoqHammer combined",
#    "firstorder eauto with *",
#])]
#df["run"] = df["run"].map({
#    "k-NN" : "k-NN",
#    "GNN No names Update new definitions" : "G2T-Anon-Update",
#    "GNN Names Update new definitions": "G2T-Named-Update",
#    "GNN No Definitions Update no definitions" : "G2T-NoDef-Frozen",
#    "Transformer GPU" : "Transformer GPU",
#    "Transformer CPU Big" : "Transformer CPU",
#    "Top 2 combined" : "G2T-Anon-Update + k-NN",
#    "CoqHammer combined" : "CoqHammer combined",
#    "firstorder eauto with *" : "firstorder",
#})
paper_models_table

In [ ]:
df = results_combined_2000_df.copy()
df = df[df["run"].isin([
    "k-NN",
    "GNN No names Update new definitions",
    "GNN Names Update new definitions",
    "GNN No Definitions Update no definitions",
    "Transformer GPU",
    "Top 2 combined",
    "CoqHammer combined",
])]
time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11"],
    theorems_to_exclude=[],
)

selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log")),
    y=alt.Y("cum_pct_solved:Q"),
    color="run:N",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "cum_solved:Q", "cum_pct_solved:Q", "seconds:Q"]
).add_params(
    selection
)

In [ ]:
df = results_combined_2000_df.copy()
#df = df[df["run"].isin([
#    "k-NN",
#    "GNN No names Update new definitions",
#    "GNN Names Update new definitions",
#    "GNN No Definitions Update no definitions",
#    "Transformer GPU",
#    "Transformer CPU Big",
#    "Top 2 combined",
#    "CoqHammer combined",
#    "firstorder eauto with *",
#])]
#df["run"] = df["run"].map({
#    "k-NN" : "k-NN",
#    "GNN No names Update new definitions" : "G2T-Anon-Update",
#    "GNN Names Update new definitions": "G2T-Named-Update",
#    "GNN No Definitions Update no definitions" : "G2T-NoDef-Frozen",
#    "Transformer GPU" : "Transformer GPU",
#    "Transformer CPU Big" : "Transformer CPU",
#    "Top 2 combined" : "G2T-Anon-Update + k-NN",
#    "CoqHammer combined" : "CoqHammer combined",
#    "firstorder eauto with *" : "firstorder",
#})
df = df[df["run"].isin(paper_models_table["run"])]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])

time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
)
display(time_df["package"].unique())
time_df = add_final_times(
    time_df,
    groupby=["run"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=10*60.0
)
display(time_df.groupby("run").last().reset_index())
print(time_df.groupby("run").last().reset_index().to_latex(index=False))
time_chart1 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title=None, scale=alt.Scale(domain=list(paper_models_table["model"]), range=list(paper_models_table["color"]))),
)

time_df = aggregate_by_time_step(
    df[df["run"].map(paper_models_table.set_index("model")["steps"])],
    time_column="model_calls",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
    up_to_percentile=0.5,
)
def correct_model_calls(row):
    if row["run"] == "k-NN":
        return row
    else:
        row["model_calls"] -= 1
        return row 
time_df = time_df.apply(correct_model_calls, axis=1)
time_chart2 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("model_calls:Q", scale=alt.Scale(type="log"), title="model calls"),
    y=alt.Y("cum_pct_solved:Q", title=None, axis=alt.Axis(labels=False, ticks=False)),
    color=alt.Color("run:N", title=None).legend(orient="top-left", fillColor="white", padding=5, offset=6, strokeColor="lightgray"),
).properties(
    height=250
)

(time_chart1 | time_chart2).resolve_scale(
    y = "shared"
)

In [ ]:
df = results_combined_2000_df.copy()
#df = df[df["run"].isin([
#    "k-NN",
#    "GNN No names Update new definitions",
#    "GNN Names Update new definitions",
#    "GNN No Definitions Update no definitions",
#    "Transformer GPU",
#    "Transformer CPU Big",
#    "Top 2 combined",
#    "CoqHammer combined",
#    "firstorder eauto with *",
#])]
#df["run"] = df["run"].map({
#    "k-NN" : "k-NN",
#    "GNN No names Update new definitions" : "G2T-Anon-Update",
#    "GNN Names Update new definitions": "G2T-Named-Update",
#    "GNN No Definitions Update no definitions" : "G2T-NoDef-Frozen",
#    "Transformer GPU" : "Transformer GPU",
#    "Transformer CPU Big" : "Transformer CPU",
#    "Top 2 combined" : "G2T-Anon-Update + k-NN",
#    "CoqHammer combined" : "CoqHammer combined",
#    "firstorder eauto with *" : "firstorder",
#})
df = df[df["run"].isin(paper_models_table["run"])]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])

time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
)
display(time_df["package"].unique())
time_df = add_final_times(
    time_df,
    groupby=["run"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=10*60.0
)
display(time_df.groupby("run").last().reset_index())
print(time_df.groupby("run").last().reset_index().to_latex(index=False))
time_chart1 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title=None).scale(domain=list(paper_models_table["model"]), range=list(paper_models_table["color"])),
    strokeDash=alt.StrokeDash("run:N", title=None).scale(domain=list(paper_models_table["model"])),
)

time_df = aggregate_by_time_step(
    df[df["run"].map(paper_models_table.set_index("model")["steps"])],
    time_column="model_calls",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
    up_to_percentile=0.5,
)
def correct_model_calls(row):
    if row["run"] == "k-NN":
        return row
    else:
        row["model_calls"] -= 1
        return row 
time_df = time_df.apply(correct_model_calls, axis=1)
time_chart2 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("model_calls:Q", scale=alt.Scale(type="log"), title="model calls"),
    y=alt.Y("cum_pct_solved:Q", title=None, axis=alt.Axis(labels=False, ticks=False)),
    color=alt.Color("run:N", title=None).scale(domain=list(paper_models_table["model"])).legend(orient="top-left", fillColor="white", padding=5, offset=6, strokeColor="lightgray"),
    strokeDash=alt.StrokeDash("run:N", title=None).scale(domain=list(paper_models_table["model"])),
).properties(
    height=250
)

(time_chart1 | time_chart2).resolve_scale(
    y = "shared"
)

In [ ]:
df = results_combined_2000_df.copy()
#df = df[df["run"].isin([
#    "k-NN",
#    "GNN No names Update new definitions",
#    "GNN Names Update new definitions",
#    "GNN No Definitions Update no definitions",
#    "Transformer GPU",
#    "Transformer CPU Big",
#    "Top 2 combined",
#    "CoqHammer combined",
#    "firstorder eauto with *",
#])]
#df["run"] = df["run"].map({
#    "k-NN" : "k-NN",
#    "GNN No names Update new definitions" : "G2T-Anon-Update",
#    "GNN Names Update new definitions": "G2T-Named-Update",
#    "GNN No Definitions Update no definitions" : "G2T-NoDef-Frozen",
#    "Transformer GPU" : "Transformer GPU",
#    "Transformer CPU Big" : "Transformer CPU",
#    "Top 2 combined" : "G2T-Anon-Update + k-NN",
#    "CoqHammer combined" : "CoqHammer combined",
#    "firstorder eauto with *" : "firstorder",
#})
display(df["run"].unique())
df = df[df["run"].isin(paper_models_table["run"])]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])
display(df["run"].unique())
display(df[(df["run"] == "Transformer-CPU") & df["solved"]])
time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
)
display(time_df["package"].unique())
display(time_df["run"].unique())
time_df = add_final_times(
    time_df,
    groupby=["run"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=10*60.0
)
display(time_df["run"].unique())
display(time_df.groupby("run").last().reset_index())
print(time_df.groupby("run").last().reset_index().to_latex(index=False))
time_chart1 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title=None).scale(domain=list(paper_models_table["model"]), scheme="category10"),
    strokeDash=alt.StrokeDash("run:N", title=None).scale(domain=list(paper_models_table["model"]), range=[[0], [3,3], [0], [3,3], [0], [3,3], [0], [0], [3,3]]),
)

time_df = aggregate_by_time_step(
    df[df["run"].map(paper_models_table.set_index("model")["steps"])],
    time_column="model_calls",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
    up_to_percentile=0.5,
)
def correct_model_calls(row):
    if row["run"] == "k-NN":
        return row
    else:
        row["model_calls"] -= 1
        return row 
time_df = time_df.apply(correct_model_calls, axis=1)
time_chart2 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("model_calls:Q", scale=alt.Scale(type="log"), title="model calls"),
    y=alt.Y("cum_pct_solved:Q", title=None, axis=alt.Axis(labels=False, ticks=False)),
    color=alt.Color("run:N", title=None).scale(domain=list(paper_models_table["model"])).legend(orient="top-left", fillColor="white", padding=5, offset=6, strokeColor="lightgray", labelLimit = 400,),
    strokeDash=alt.StrokeDash("run:N", title=None).scale(domain=list(paper_models_table["model"])),
).properties(
    height=250
)

chart = (time_chart1 | time_chart2).resolve_scale(
    y = "shared"
)
chart.save("paper_images/model_times.png", scale_factor=4)
chart

In [ ]:
df = results_combined_2000_df.copy()
#df = df[df["run"].isin([
#    "k-NN",
#    "GNN No names Update new definitions",
#    "GNN Names Update new definitions",
#    "GNN No Definitions Update no definitions",
#    "Transformer GPU",
#    "Transformer CPU Big",
#    "Top 2 combined",
#    "CoqHammer combined",
#    "firstorder eauto with *",
#])]
#df["run"] = df["run"].map({
#    "k-NN" : "k-NN",
#    "GNN No names Update new definitions" : "G2T-Anon-Update",
#    "GNN Names Update new definitions": "G2T-Named-Update",
#    "GNN No Definitions Update no definitions" : "G2T-NoDef-Frozen",
#    "Transformer GPU" : "Transformer GPU",
#    "Transformer CPU Big" : "Transformer CPU",
#    "Top 2 combined" : "G2T-Anon-Update + k-NN",
#    "CoqHammer combined" : "CoqHammer combined",
#    "firstorder eauto with *" : "firstorder",
#})
df = df[df["run"].isin(paper_models_table["run"])]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])

time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
)
time_df = add_final_times(
    time_df,
    groupby=["run"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=10*60.0
)

time_chart1 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title="Model", scale=alt.Scale(domain=list(paper_models_table["model"]), range=list(paper_models_table["venn_color"]))),
)

time_df = aggregate_by_time_step(
    df[df["run"].map(paper_models_table.set_index("model")["steps"])],
    time_column="model_calls",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
    up_to_percentile=0.5,
)
def correct_model_calls(row):
    if row["run"] == "k-NN":
        return row
    else:
        row["model_calls"] -= 1
        return row 
time_df = time_df.apply(correct_model_calls, axis=1)
time_chart2 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("model_calls:Q", scale=alt.Scale(type="log"), title="model calls"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title=None).legend(orient="top-left", fillColor="white", padding=5, strokeColor="lightgray"),
)

time_chart1 | time_chart2

In [ ]:
df = results_combined_2000_df.copy()
df = df[df["run"].str.startswith("CoqHammer")]
time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11"],
    theorems_to_exclude=[],
)
display(time_df["run"].unique())
time_df["run"] = time_df["run"].map(lambda run: run.split("CoqHammer ")[1])
time_df = add_final_times(
    time_df,
    groupby=["run"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=10*60.0
)
chart = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title=None).legend(orient="top-left", fillColor="white", padding=5, strokeColor="lightgray", offset=6, labelLimit = 400,),
)
chart.save("paper_images/coqhammer_times.png", scale_factor=4)
chart

In [ ]:
# chart of speeds

df = results_2000_df.copy()

paper_table_copy = paper_models_table.copy()
paper_table_copy = paper_models_table[paper_models_table["steps"]]
df = df[df["run"].isin(paper_table_copy["run"])]
df["run"] = df["run"].map(paper_table_copy.set_index("run")["model"])

df = df[df["messages"].notna()]
df["messages"] = df["messages"].astype(int)
df = df[df["steps"].notna()]
df["steps"] = df["steps"].astype(int)
df["tactics_executed_per_second"] = df["steps"] / df["time"]
df["log_tactics_executed_per_second"] = np.log(df["tactics_executed_per_second"])
df["model_calls_per_second"] = df["messages"] / df["time"]
df["log_model_calls_per_second"] = np.log(df["model_calls_per_second"])
df = df[df["tactics_executed_per_second"].notna() & df["model_calls_per_second"].notna()]

# sort according to order
ordered_titles = list(df.groupby("run")["log_tactics_executed_per_second"].median().sort_values().index)
colors = list(paper_table_copy.set_index("model")["color"][ordered_titles])
print(ordered_titles, colors)

chart2 = alt.Chart(df).mark_line().transform_density(
    'log_tactics_executed_per_second',
    as_=['xlog_tactics_executed_per_second', 'density'],
    groupby=['run']
).transform_calculate(
    xx="exp(datum.xlog_tactics_executed_per_second)"
).encode(
    x=alt.X("xx:Q", scale=alt.Scale(type="log"), title="Tactics executed per second"),
    y=alt.X('density:Q', title="Density", axis=None),
    color=alt.Color("run:N", title="Model")
).properties(
    height=100,
)

chart1 = alt.Chart(df).mark_line().transform_density(
    'log_model_calls_per_second',
    as_=['xlog_model_calls_per_second', 'density'],
    groupby=['run']
).transform_calculate(
    xx="exp(datum.xlog_model_calls_per_second)"
).encode(
    x=alt.X("xx:Q", scale=alt.Scale(type="log"), title="Model calls per second"),
    y=alt.X('density:Q', title="Density", axis=None),
    color=alt.Color("run:N", title=None, legend=alt.Legend(orient="top",), scale=alt.Scale(domain=ordered_titles, range=colors)),
).properties(
    height=100,
)

chart = chart1 | chart2
chart.save("paper_images/model_speeds.png", scale_factor=4)
chart

In [ ]:
g2t_paper_models_table = pd.DataFrame([
    #{"run": "Top 2 combined", "model": "G2T-Anon-Update + k-NN", "venn_color": "lightgrey", "color": "#9d755d", "steps": False},
    #{"run": "k-NN", "model": "k-NN", "venn_color": "#7E00FF", "color": "#b279a2", "steps": True},
    {"run": "GNN No names Update new definitions", "model": "G2T-Anon-Update", "venn_color": "#FF5400", "color": "#e45756", "steps": True},
    {"run": "GNN Names Update new definitions", "model": "G2T-Named-Update", "venn_color": "lightgrey", "color": "#4c78a8", "steps": True},
    {"run": "GNN Names Update all definitions", "model": "G2T-Named-Recalc", "venn_color": "lightgrey", "color": "#ff9da6", "steps": True},
    {"run": "GNN No Definitions Update no definitions", "model": "G2T-NoDef-Frozen", "venn_color": "lightgrey", "color": "#f58518", "steps": True},
    {"run": "GNN No names Update all definitions", "model": "G2T-Anon-Recalc", "venn_color": "lightgrey", "color": "#eeca3b", "steps": True},
    {"run": "GNN Names Update no definitions", "model": "G2T-Named-Frozen", "venn_color": "lightgrey", "color": "#b279a2", "steps": True},
    {"run": "GNN No names Update no definitions", "model": "G2T-Anon-Frozen", "venn_color": "#FF5400", "color": "#54a24b", "steps": True},
    
    
    
    #{"run": "Transformer GPU", "model": "Transformer-GPU", "venn_color": "#F7FF00", "color": "#eeca3b", "steps": True},
    #{"run": "Transformer CPU Big", "model": "Transformer-CPU", "venn_color": "lightgrey", "color": "#ff9da6", "steps": True},
    #{"run": "CoqHammer combined", "model": "CoqHammer combined", "venn_color": "#00A61F", "color": "#54a24b", "steps": False},
    #{"run": "firstorder eauto with *", "model": "firstorder", "venn_color": "lightgrey", "color": "#72b7b2", "steps": False},
])

df = results_combined_2000_df.copy()
df = df[df["run"].isin(g2t_paper_models_table["run"])]
df["run"] = df["run"].map(g2t_paper_models_table.set_index("run")["model"])

time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
)
display(time_df["package"].unique())
time_df = add_final_times(
    time_df,
    groupby=["run"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=10*60.0
)

time_chart1 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title=None, scale=alt.Scale(domain=list(g2t_paper_models_table["model"]), range=list(g2t_paper_models_table["color"]))),
)

chart = time_chart1
chart.save("paper_images/g2t-times.png", scale_factor=4)
chart

In [ ]:
df = results_combined_500_each_df.copy()

df = df[df["package"].str.contains("tlc")]

df = df[df["run"].isin(paper_models_table["run"])]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])

df1 = df.copy()

time_df = aggregate_by_time_step(
    df1,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
)
display(time_df["package"].unique())
time_df = add_final_times(
    time_df,
    groupby=["run"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=10*60.0
)

time_chart1 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title=None, scale=alt.Scale(domain=list(paper_models_table["model"]), range=list(paper_models_table["color"]))),
).properties(
    title="tlc (all test theorems)"
)

df2 = df.copy()
df2["skip_axiom"] = ~df2["found_proof"].isna() & (df2["found_proof"].str.contains("skip_axiom") | df2["found_proof"].str.contains("No_Empty_admitted"))
bad_skip_axiom_thms = df2[df2["skip_axiom"]]["theorem"].unique()
df2 = df2[~df2["theorem"].isin(bad_skip_axiom_thms)]

time_df = aggregate_by_time_step(
    df2,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
)
display(time_df["package"].unique())
time_df = add_final_times(
    time_df,
    groupby=["run"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=10*60.0
)

time_chart2 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title=None, axis=alt.Axis(labels=False, ticks=False)),
    color=alt.Color("run:N", title=None).legend(orient="top-left", fillColor="white", padding=5, offset=6, strokeColor="lightgray", labelLimit = 400,),
).properties(
    title="tlc (safe test theorems)",
    height=250
)

chart =(time_chart1 | time_chart2).resolve_scale(
    y = "shared"
)
chart.save("paper_images/tlc-axiom-discussion-times.png", scale_factor=4)
chart

## Cheating stats for paper

In [ ]:
df = results_500_each_df.copy()
df = df[df["run"].isin(paper_models_table["run"])]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])

bad_axioms = ["skip_axiom", "ignore_generator_proofs", "undefined", "No_Empty_admitted"]
bad_axioms_by_package = {"tlc": "skip_axiom", "haskell":"undefined", "hott": "No_Empty_admitted"}
df["cheated"] = False
df["axiom"] = ""
for axiom in bad_axioms:
    df["cheated"] |= df["found_proof"].str.contains(axiom)
    #df["axiom"] += axiom * df["found_proof"].str.contains(axiom)
df["cheated"] &= ~df["found_proof"].isna()

df["package"] = df["package"].map(lambda x: x.split("-")[1].split(".")[0])

df = df.pivot_table(index=["package", "run"], columns=[], values="cheated", aggfunc=["sum", "mean"])
df.columns = df.columns.droplevel(1)
df = df.reset_index()
df = df.rename(columns={"run": "solver", "sum": "count", "mean": "percent"})
df["percent"] = df["percent"]*100
df = df[df["percent"] > 0]
df["axiom"] = df["package"].map(bad_axioms_by_package)
df = df[["package", "axiom", "solver", "count", "percent"]]
#df = df.set_index(["package", "axiom", "solver"])
display(df)
df["axiom"] = "\\verb|" + df["axiom"] + "|"
df["percent"] = df["percent"].apply(lambda p: f'{p:.1f}\\%')
print(df.to_latex(index=False))


## Plots for internal talk

In [ ]:
df = results_combined_2000_df.copy()
df = df[df["run"].isin([
    "k-NN",
    "GNN No names Update new definitions",
    "GNN No Definitions Update no definitions",
    "Transformer GPU",
    "Top 2 combined",
    "CoqHammer combined",
    "firstorder eauto with *"
])]
df["run"] = df["run"].map({
    "k-NN" : "k-NN (ML baseline)",
    "GNN No names Update new definitions" : "G2T-UpdateNewDefs",
    "GNN No Definitions Update no definitions" : "G2T-NoDef",
    "Transformer GPU" : "Transformer GPU (ML baseline)",
    "Top 2 combined" : "G2T-UpdateNewDefs + k-NN",
    "CoqHammer combined" : "CoqHammer (symbolic baseline)",
    "firstorder eauto with *" : "firstorder (symbolic baseline)"
})
time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11"],
    theorems_to_exclude=[],
)

selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="Time (seconds)"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title="Model"),
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "cum_solved:Q", "cum_pct_solved:Q", "seconds:Q"]
).add_params(
    selection
).properties(
    title="Test theorems solved over time"
)

## Main plots for 500 each

In [ ]:
time_df = aggregate_by_time_step(
    results_2000_df,
    time_column="seconds",
    groupby=["run", "package"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
)

selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(time_df).mark_line().encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log")),
    y="cum_pct_solved:Q",
    color="run",
    row="package",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "cum_solved:Q", "cum_pct_solved:Q", "seconds:Q"]
).add_params(
    selection
)

In [ ]:
time_df = aggregate_by_time_step(
    results_2000_df,
    time_column="seconds",
    groupby=["run", "package"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
)

for package, df in time_df.groupby("package"):
    selection = alt.selection_point(fields=['run'], bind='legend')
    chart = alt.Chart(df).mark_line(clip=True).encode(
        x=alt.X("seconds:Q", scale=alt.Scale(type="log")),
        y=alt.Y("cum_pct_solved:Q"),
        color="run",
        opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
        tooltip=["run:N", "cum_solved:Q", "cum_pct_solved:Q", "seconds:Q"],
    ).add_params(
        selection
    ).properties(
        title=f"{package}",
    )
    display(chart)

## Grid plots for paper

In [ ]:
df = results_combined_500_each_df.copy()
df = df[df["run"].isin([
    "k-NN",
    "GNN No names Update new definitions",
    "CoqHammer combined",
])]

time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run", "package"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
)
time_df = add_final_times(
    time_df,
    groupby=["run", "package"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=5*60.0
)
chart = alt.Chart(time_df).mark_line().encode(
    x=alt.X("seconds:Q"),
    y="cum_pct_solved:Q",
    color="run",
).properties(
    width=150,
    height=150
).facet(
    facet='package:N',
    columns=4
)

chart

In [ ]:
df = results_combined_500_each_df.copy()
runs = [
    "k-NN",
    "GNN No names Update new definitions",
    "CoqHammer combined",
]
df = df[df["run"].isin(runs)]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])

time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run", "package"],
    packages_to_exclude=["coq-bytestring.0.9.0", "coq-printf.2.0.0", "coq-ceres.0.4.0", "coq-haskell.1.0.0", "coq-poltac.0.8.11"],
    theorems_to_exclude=[],
)
time_df = add_final_times(
    time_df,
    groupby=["run", "package"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=5*60.0
)

# sort according to order
ordered_titles = list(paper_models_table.set_index("run")["model"][runs])
colors = list(paper_models_table.set_index("run")["color"][runs])
print(ordered_titles, colors)

chart = alt.Chart(time_df).mark_line().encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title=None, scale=alt.Scale(domain=ordered_titles, range=colors)).legend(orient="top-left", fillColor="white", padding=5, offset=6, strokeColor="lightgray"),
).properties(
    width=150,
    height=150
).facet(
    facet=alt.Facet('package:N', title=None),
    columns=4,
    spacing={"row": 20, "column": 10},
)

chart

In [ ]:
df = results_combined_500_each_df.copy()

df = df[df["run"].isin(paper_models_table["run"])]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])

time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run", "package"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
)
time_df = add_final_times(
    time_df,
    groupby=["run", "package"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=5*60.0
)

# sort according to order
ordered_titles = list(paper_models_table.set_index("run")["model"])
colors = list(paper_models_table.set_index("run")["color"])
print(ordered_titles, colors)

chart = alt.Chart(time_df).mark_line().encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title=None, scale=alt.Scale(domain=ordered_titles, range=colors)).legend(orient="none", fillColor="white", padding=5, legendX=485, legendY=785, strokeColor="lightgray", labelLimit = 400,),
).properties(
    width=150,
    height=150
).facet(
    facet=alt.Facet('package:N', title=None),
    columns=4,
    spacing={"row": 20, "column": 10},
)

chart
chart.save("paper_images/grid_plot_all_packages_all_models.png", scale_factor=4)
chart

In [ ]:
df = results_combined_500_each_df.copy()
runs = [
    "k-NN",
    "GNN No names Update new definitions",
    "CoqHammer combined",
]
df = df[df["run"].isin(runs)]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])

time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run", "package"],
    packages_to_exclude=["coq-bytestring.0.9.0", "coq-printf.2.0.0"],
    theorems_to_exclude=[],
)
time_df = add_final_times(
    time_df,
    groupby=["run", "package"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=5*60.0
)
time_df["package"] = time_df["package"].apply(lambda x: x.split(".")[0].split("coq-")[1])

# sort according to order
ordered_titles = list(paper_models_table.set_index("run")["model"][runs])
colors = list(paper_models_table.set_index("run")["color"][runs])
print(ordered_titles, colors)

chart = alt.Chart(time_df).mark_line().encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds", axis=alt.Axis(grid=False)),
    y=alt.Y("cum_pct_solved:Q", title="Pass Rate").scale(domain=(0,1.0)).axis(tickCount=6),
    #color=alt.Color("run:N", title=None, scale=alt.Scale(domain=ordered_titles, range=colors)).legend(orient="top-left", fillColor="white", padding=5, offset=6, strokeColor="lightgray"),
    color=alt.Color("run:N", title=None, scale=alt.Scale(domain=ordered_titles, range=colors)).legend(orient="top", offset=6),
).properties(
    width=100,
    height=100,
).facet(
    facet=alt.Facet('package:N', title=None, header=alt.Header(labelPadding=-15)),
    columns=5,
    spacing={"row": 10, "column": 10},
)

chart.save("paper_images/grid_plot_with_ch_15_packages.png", scale_factor=4)
chart

In [ ]:
df = results_combined_500_each_df.copy()
df = df[df["run"].isin([
    "k-NN",
    "GNN No names Update new definitions",
    "CoqHammer combined",
])]

time_df = aggregate_by_time_step(
    df,
    time_column="model_calls",
    groupby=["run", "package"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
)

chart = alt.Chart(time_df).mark_line().encode(
    x=alt.X("model_calls:Q", scale=alt.Scale(type="log")),
    y="cum_pct_solved:Q",
    color="run"
).properties(
    width=150,
    height=150
).facet(
    facet='package:N',
    columns=4
)

chart

In [ ]:
df = results_combined_500_each_df.copy()
df = df[df["run"].isin([
    "k-NN",
    "GNN No names Update new definitions",
    "CoqHammer combined",
])]

time_df = aggregate_by_time_step(
    df,
    time_column="tactics_executed",
    groupby=["run", "package"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
)

chart = alt.Chart(time_df).mark_line().encode(
    x=alt.X("tactics_executed:Q", scale=alt.Scale(type="log")),
    y="cum_pct_solved:Q",
    color="run",
    tooltip=["run:N", "cum_solved:Q", "cum_pct_solved:Q", "tactics_executed:Q"],
).properties(
    width=150,
    height=150
).facet(
    facet='package:N',
    columns=4
)

chart

In [ ]:
df = results_combined_500_each_df.copy()
df = df[df["run"].isin([
    "k-NN",
    "GNN No names Update new definitions",
    "CoqHammer combined",
])]

time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run", "package"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
)

chart_base = alt.Chart(time_df).mark_line().encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log")),
    y="cum_pct_solved:Q",
    color="run",
    tooltip=["run:N", "cum_solved:Q", "cum_pct_solved:Q", "seconds:Q"],
).properties(
    width=150,
    height=150
)

chart_regression = chart_base.transform_regression(
    'seconds', 'cum_pct_solved',
    method="log",
    groupby=["run", "package"]
).mark_line(strokeDash=[3,3])

chart = (chart_base + chart_regression).facet(
    facet='package:N',
    columns=4
)

chart

In [ ]:
df = results_combined_500_each_df.copy()
df = df[df["run"].isin([
    "k-NN",
    "GNN No names Update new definitions",
    "CoqHammer combined",
])]

time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run", "package"],
    packages_to_exclude=[],
    theorems_to_exclude=[],
)
time_df = time_df[time_df["seconds"] >= 10.0]

chart_base = alt.Chart(time_df).mark_line().encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log")),
    y="cum_pct_solved:Q",
    color="run",
    tooltip=["run:N", "cum_solved:Q", "cum_pct_solved:Q", "seconds:Q"],
).properties(
    width=150,
    height=150
)

chart_regression = chart_base.transform_regression(
    'seconds', 'cum_pct_solved',
    method="log",
    groupby=["run", "package"]
).mark_line(strokeDash=[3,3])

chart = (chart_base + chart_regression).facet(
    facet='package:N',
    columns=4
)

chart

## 500 results table for paper

In [ ]:
df = results_500_each_df.copy()



## Lemma distance

In [ ]:
lemma_distance_df = pd.read_csv(lemma_dist_data_path, sep="\t", names=["package", "theorem", "lemma_distance"], dtype=str)
lemma_distance_df["lemma_distance"] = lemma_distance_df["lemma_distance"].astype(int)
lemma_distance_df

In [ ]:
lemma_size_df = pd.read_csv(lemma_size_data_path, sep="\t", names=["theorem", "lemma_size"], dtype=str)
lemma_size_df["lemma_size"] = lemma_size_df["lemma_size"].astype(int)
lemma_size_df

In [ ]:
new_tactic_df = pd.read_csv(lemma_new_tactics_data_path, sep="\t", names=["theorem"], dtype=str)
new_tactic_df["new_tactics_in_original"] = True
new_tactic_df

no_new_tactic_df = pd.read_csv(lemma_no_new_tactics_data_path, sep="\t", names=["theorem"], dtype=str)
no_new_tactic_df["new_tactics_in_original"] = False
no_new_tactic_df

new_tactic_df = pd.concat([new_tactic_df, no_new_tactic_df])
new_tactic_df

In [ ]:
df = results_500_each_df.copy()
df.pivot_table(values="solved", columns="run", index="package", aggfunc="sum")[["GNN Names Update new definitions", "GNN No names Update new definitions", "k-NN"]]

In [ ]:
df = results_2000_df.copy()
df = pd.merge(df, lemma_distance_df, on="theorem", how="outer")
df["in_lemma_distance_data"] = ~df["package_y"].isna()
df = df[df["run"].isin(["GNN Names Update new definitions", "GNN No names Update new definitions", "k-NN"])]
print(df.pivot_table(values="solved", columns="in_lemma_distance_data", index="run", aggfunc="count", margins=True))

In [ ]:
df = results_500_each_df.copy()
df = pd.merge(df, lemma_distance_df, on="theorem", how="outer")
df["in_lemma_distance_data"] = ~df["package_y"].isna()
print(df.pivot_table(values="solved", columns=["run", "in_lemma_distance_data"], index=["package_y", ], aggfunc="sum")[["GNN Names Update new definitions", "GNN No names Update new definitions", "k-NN"]].astype(int))

In [ ]:
df = results_500_each_df.copy()
df = pd.merge(df, lemma_distance_df, on="theorem", how="outer")
df["in_lemma_distance_data"] = ~df["package_y"].isna()
df = df[df["run"].isin(["GNN Names Update new definitions", "GNN No names Update new definitions", "k-NN"])]
print(df.pivot_table(values="solved", columns=["run", "in_lemma_distance_data"], index="package_x", aggfunc="sum", margins=True))

In [ ]:
df = results_500_each_df.copy()
df = pd.merge(df, lemma_distance_df, on="theorem", how="outer")
df["in_lemma_distance_data"] = ~df["package_y"].isna()
df = df[df["run"].isin(["k-NN"])]
print(df.pivot_table(values="solved", columns=["in_lemma_distance_data"], index="package_x", aggfunc="count", margins=True))

In [ ]:
df = results_2000_df.copy()
df = pd.merge(df, lemma_distance_df, on="theorem", how="outer")
df["in_lemma_distance_data"] = ~df["package_y"].isna()
df = df[df["run"].isin(["GNN Names Update new definitions", "GNN No names Update new definitions", "k-NN"])]
print(df.pivot_table(values="solved", columns="in_lemma_distance_data", index="run", aggfunc="count", margins=True))

### Lemma distance plots

In [ ]:
df = results_2000_df.copy()
df = pd.merge(df, lemma_distance_df, on="theorem", how="inner")
df = df.sort_values("lemma_distance")
df["cum_solved"] = df.groupby(["run"])["solved"].transform("cumsum")
df["lemma_distance"] += 1
selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(df).mark_line().encode(
    x=alt.X("lemma_distance:Q", scale=alt.Scale(type="log")),
    y="cum_solved:Q",
    color="run:N",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "solved:Q", "time:Q"]
).add_params(
    selection
)


In [ ]:
df = results_2000_df.copy()
df = pd.merge(df, lemma_distance_df, on="theorem", how="inner")
df = df.sort_values("lemma_distance")
df["cum_solved"] = df.groupby(["run"])["solved"].transform("cumsum")
df["lemma_distance"] += 1
selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(df).mark_line(clip=True).encode(
    x=alt.X("lemma_distance:Q", scale=alt.Scale(type="log", domain=[1, 10])),
    y=alt.Y("cum_solved:Q").scale(domain=[0, 250]),
    color="run:N",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "solved:Q", "time:Q"]
).add_params(
    selection
)


In [ ]:
# for paper?
df = results_2000_df.copy()
df = df[~df["run"].isin(["coq-hott.8.11", "coq-tlc.20200328"])]
df = pd.merge(df, lemma_distance_df, on="theorem", how="inner")
df = df.sort_values("lemma_distance")
df = df.groupby(["run", "lemma_distance"])[["solved"]].mean().reset_index()
selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(df).mark_line(clip=True).encode(
    x=alt.X("lemma_distance:Q", scale=alt.Scale(domain=[0, 10])),
    y=alt.Y("solved:Q"),
    color="run:N",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "solved:Q"]
).add_params(
    selection
)


In [ ]:
# for paper?
df = results_combined_2000_df.copy()
df = pd.merge(df, lemma_distance_df, on=["theorem", "package"], how="inner")
df = df[df["lemma_distance"] <= 0]

df = df[df["run"].isin(paper_models_table["run"])]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])

display(df)
time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
)
display(time_df["package"].unique())
time_df = add_final_times(
    time_df,
    groupby=["run"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=10*60.0
)

time_chart1 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title=None, scale=alt.Scale(domain=list(paper_models_table["model"]), range=list(paper_models_table["color"]))),
)

time_df = aggregate_by_time_step(
    df[df["run"].map(paper_models_table.set_index("model")["steps"])],
    time_column="model_calls",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
    up_to_percentile=0.5,
)
def correct_model_calls(row):
    if row["run"] == "k-NN":
        return row
    else:
        row["model_calls"] -= 1
        return row 
time_df = time_df.apply(correct_model_calls, axis=1)
time_chart2 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("model_calls:Q", scale=alt.Scale(type="log"), title="model calls"),
    y=alt.Y("cum_pct_solved:Q", title=None, axis=alt.Axis(labels=False, ticks=False)),
    color=alt.Color("run:N", title=None).legend(orient="top-left", fillColor="white", padding=5, offset=6, strokeColor="lightgray"),
).properties(
    height=250
)

(time_chart1 | time_chart2).resolve_scale(
    y = "shared"
)


In [ ]:
# for paper?
df = results_combined_2000_df.copy()
df = pd.merge(df, lemma_distance_df, on=["theorem", "package"], how="inner")
df = df[df["lemma_distance"] <= 3]

df = df[df["run"].isin(paper_models_table["run"])]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])

display(df)
time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
)
display(time_df["package"].unique())
time_df = add_final_times(
    time_df,
    groupby=["run"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=10*60.0
)

time_chart1 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title=None, scale=alt.Scale(domain=list(paper_models_table["model"]), range=list(paper_models_table["color"]))),
)

time_df = aggregate_by_time_step(
    df[df["run"].map(paper_models_table.set_index("model")["steps"])],
    time_column="model_calls",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
    up_to_percentile=0.5,
)
def correct_model_calls(row):
    if row["run"] == "k-NN":
        return row
    else:
        row["model_calls"] -= 1
        return row 
time_df = time_df.apply(correct_model_calls, axis=1)
time_chart2 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("model_calls:Q", scale=alt.Scale(type="log"), title="model calls"),
    y=alt.Y("cum_pct_solved:Q", title=None, axis=alt.Axis(labels=False, ticks=False)),
    color=alt.Color("run:N", title=None).legend(orient="top-left", fillColor="white", padding=5, offset=6, strokeColor="lightgray"),
).properties(
    height=250
)

(time_chart1 | time_chart2).resolve_scale(
    y = "shared"
)


In [ ]:
# for paper?
df = results_combined_2000_df.copy()
df = pd.merge(df, lemma_distance_df, on=["theorem", "package"], how="inner")
df = df[df["lemma_distance"] > 10]

df = df[df["run"].isin(paper_models_table["run"])]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])

display(df)
time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
)
display(time_df["package"].unique())
time_df = add_final_times(
    time_df,
    groupby=["run"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=10*60.0
)

time_chart1 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title=None, scale=alt.Scale(domain=list(paper_models_table["model"]), range=list(paper_models_table["color"]))),
)

time_df = aggregate_by_time_step(
    df[df["run"].map(paper_models_table.set_index("model")["steps"])],
    time_column="model_calls",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
    up_to_percentile=0.5,
)
def correct_model_calls(row):
    if row["run"] == "k-NN":
        return row
    else:
        row["model_calls"] -= 1
        return row 
time_df = time_df.apply(correct_model_calls, axis=1)
time_chart2 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("model_calls:Q", scale=alt.Scale(type="log"), title="model calls"),
    y=alt.Y("cum_pct_solved:Q", title=None, axis=alt.Axis(labels=False, ticks=False)),
    color=alt.Color("run:N", title=None).legend(orient="top-left", fillColor="white", padding=5, offset=6, strokeColor="lightgray"),
).properties(
    height=250
)

(time_chart1 | time_chart2).resolve_scale(
    y = "shared"
)


In [ ]:
df = results_2000_df.copy()

#df["skip_axiom"] = ~df["found_proof"].isna() & (df["found_proof"].str.contains("skip_axiom") | df["found_proof"].str.contains("No_Empty_admitted"))
#bad_skip_axiom_thms = df[df["skip_axiom"]]["theorem"].unique()
#df = df[~df["theorem"].isin(bad_skip_axiom_thms)]

df = pd.merge(df, lemma_distance_df, on="theorem", how="inner")
df = df.groupby(["run", "lemma_distance"])[["solved"]].mean().reset_index()
df["lemma_distance"] += 1
df["log_lemma_distance"] = np.log(df["lemma_distance"])
selection = alt.selection_point(fields=['run'], bind='legend')
chart = alt.Chart(df).mark_line().encode(
    x=alt.X("log_lemma_distance:Q",  bin=alt.Bin(maxbins=10)),
    y="mean(solved):Q",
    color="run:N",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "mean(solved):Q", "sum(solved):Q"]
).add_params(
    selection
)
display(chart)


In [ ]:
df = results_2000_df.copy()

#df["skip_axiom"] = ~df["found_proof"].isna() & (df["found_proof"].str.contains("skip_axiom") | df["found_proof"].str.contains("No_Empty_admitted"))
#bad_skip_axiom_thms = df[df["skip_axiom"]]["theorem"].unique()
#df = df[~df["theorem"].isin(bad_skip_axiom_thms)]

df = pd.merge(df, lemma_distance_df, on="theorem", how="inner")
df = df.groupby(["run", "lemma_distance"])[["solved"]].sum().reset_index()
df["lemma_distance"] += 1
df["log_lemma_distance"] = np.log(df["lemma_distance"])
selection = alt.selection_point(fields=['run'], bind='legend')
chart = alt.Chart(df).mark_line().encode(
    x=alt.X("log_lemma_distance:Q",  bin=alt.Bin(maxbins=10)),
    y="sum(solved):Q",
    color="run:N",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "mean(solved):Q", "sum(solved):Q"]
).add_params(
    selection
)
display(chart)


In [ ]:
# for paper?
df = results_2000_df.copy()

#df["skip_axiom"] = ~df["found_proof"].isna() & (df["found_proof"].str.contains("skip_axiom") | df["found_proof"].str.contains("No_Empty_admitted"))
#bad_skip_axiom_thms = df[df["skip_axiom"]]["theorem"].unique()
#df = df[~df["theorem"].isin(bad_skip_axiom_thms)]

df = pd.merge(df, lemma_distance_df, on="theorem", how="inner")
df["lemma_distance"] += 1
df["log_lemma_distance"] = np.log(df["lemma_distance"])
df1 = df[df["solved"]].copy()
df1["atype"] = "number solved"
selection = alt.selection_point(fields=['atype'], bind='legend')
chart1 = alt.Chart(df1).mark_line().transform_density(
    'log_lemma_distance',
    counts=True,
    as_=['xlog_lemma_distance', 'nsolved'],
    groupby=['run', "atype"]
).transform_calculate(
    xx="exp(datum.xlog_lemma_distance)"
).encode(
    x=alt.X("xx:Q", title="Lemma distance").scale(type="log", domain=[1, 1000]),
    y=alt.Y('nsolved:Q', title="Num solved (smoothed)"),
    color=alt.Color("run:N", title=None),
    strokeDash=alt.StrokeDash("atype:N", title=None).scale(domain=["number solved", "total count"]),
    opacity=alt.condition(selection, alt.value(1), alt.value(0.1)),
    tooltip=["atype:N"],
).add_params(
    selection
)

df2 = df[df["run"] == "k-NN"].copy()
df2["atype"] = "total count"
chart2 = alt.Chart(df2).mark_line().transform_density(
    'log_lemma_distance',
    counts=True,
    as_=['xlog_lemma_distance', 'nsolved'],
    groupby=['run', "atype"]
).transform_calculate(
    xx="exp(datum.xlog_lemma_distance)"
).encode(
    x=alt.X("xx:Q", title="Lemma distance").scale(type="log", domain=[1, 1000]),
    y=alt.Y('nsolved:Q', title="Num solved (smoothed)"),
    strokeDash=alt.StrokeDash("atype:N", title=None).scale(domain=["number solved", "total count"]),
    opacity=alt.condition(selection, alt.value(1), alt.value(0.1)),
    tooltip=["atype:N"],
).add_params(
    selection
)
display(chart1 + chart2)

In [ ]:
for k, df in results_500_each_df.copy().groupby("package"):
    display(k)
    df = pd.merge(df, lemma_distance_df, on="theorem", how="inner")
    df = df.groupby(["run", "lemma_distance"])[["solved"]].sum().reset_index()
    df["lemma_distance"] += 1
    df["log_lemma_distance"] = np.log(df["lemma_distance"])
    selection = alt.selection_point(fields=['run'], bind='legend')
    if len(df):
        chart = alt.Chart(df).mark_line().encode(
            x=alt.X("log_lemma_distance:Q",  bin=alt.Bin(maxbins=10)),
            y="sum(solved):Q",
            color="run:N",
            opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
            tooltip=["run:N", "mean(solved):Q", "sum(solved):Q"]
        ).add_params(
            selection
        )
        display(chart)


In [ ]:
for k, df in results_500_each_df.copy().groupby("package"):
    if "hott" in k:
        continue
    display(k)
    df = pd.merge(df, lemma_distance_df, on="theorem", how="inner")
    df["lemma_distance"] += 1  # to prevent log(0)
    # smooth out the data
    # first take mean over all points with same lemma distance
    df["solved"] = df.groupby(["run", "lemma_distance"])[["solved"]].transform("mean")
    # use ewm to average data
    df = df.sort_values("lemma_distance")
    #df["smooth_solved"] = df.groupby("run")["solved"].transform(lambda x: x.ewm(halflife=100).mean())
    df["smooth_solved"] = df.groupby("run")["solved"].transform(lambda x: x.rolling(10).mean())
    # replace with mean
    df = df.groupby(["run", "lemma_distance"])[["smooth_solved"]].mean().reset_index()

    selection = alt.selection_point(fields=['run'], bind='legend')
    chart = alt.Chart(df).mark_line().encode(
        x=alt.X("lemma_distance:Q", scale=alt.Scale(type="log")),
        y="smooth_solved:Q",
        color="run:N",
        opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
        tooltip=["run:N", "mean(smooth_solved):Q", "sum(smooth_solved):Q"]
    ).add_params(
        selection
    )
    display(chart)


In [ ]:
df = results_500_each_df.copy()
df = pd.merge(df, lemma_distance_df, on="theorem", how="inner")
df = pd.merge(df, lemma_size_df, on="theorem", how="inner")
#df = df[df["lemma_size"] == 1]
df["lemma_distance"] += 1  # to prevent log(0)
# smooth out the data
# first take mean over all points with same lemma distance
df["solved"] = df.groupby(["run", "lemma_distance"])[["solved"]].transform("mean")
# use ewm to average data
df = df.sort_values("lemma_distance")
#df["smooth_solved"] = df.groupby("run")["solved"].transform(lambda x: x.ewm(halflife=100).mean())
df["smooth_solved"] = df.groupby("run")["solved"].transform(lambda x: x.rolling(100).mean())
# replace with mean
df = df.groupby(["run", "lemma_distance"])[["smooth_solved"]].mean().reset_index()

selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(df).mark_line().encode(
    x=alt.X("lemma_distance:Q", scale=alt.Scale(type="log")),
    y="smooth_solved:Q",
    color="run:N",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "mean(smooth_solved):Q", "sum(smooth_solved):Q"]
).add_params(
    selection
)


In [ ]:
df = results_500_each_df.copy()
df = pd.merge(df, lemma_distance_df[["theorem", "lemma_distance"]], on="theorem", how="inner")
df = df[df["lemma_distance"] < 100]
df.pivot_table(values="solved", columns="run", index="package", aggfunc="sum")[["GNN Names Update new definitions", "GNN No names Update new definitions", "k-NN"]]

In [ ]:
df = results_500_each_df.copy()
df = pd.merge(df, lemma_size_df, on="theorem", how="inner")
df = df.sort_values("lemma_size")
df["cum_solved"] = df.groupby(["run"])["solved"].transform("cumsum")
#df["lemma_size"] += 1
selection = alt.selection_point(fields=['run'], bind='legend')

alt.Chart(df).mark_line().encode(
    x=alt.X("lemma_size:Q", scale=alt.Scale(type="log")),
    y="cum_solved:Q",
    color="run:N",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "solved:Q", "time:Q"]
).add_params(
    selection
)


In [ ]:
df = results_2000_df.copy()

#df["skip_axiom"] = ~df["found_proof"].isna() & (df["found_proof"].str.contains("skip_axiom") | df["found_proof"].str.contains("No_Empty_admitted"))
#bad_skip_axiom_thms = df[df["skip_axiom"]]["theorem"].unique()
#df = df[~df["theorem"].isin(bad_skip_axiom_thms)]

df = pd.merge(df, lemma_size_df, on="theorem", how="inner")
df = df.groupby(["run", "lemma_size"])[["solved"]].sum().reset_index()
df["log_lemma_size"] = np.log(df["lemma_size"])
selection = alt.selection_point(fields=['run'], bind='legend')
chart = alt.Chart(df).mark_line().encode(
    x=alt.X("log_lemma_size:Q",  bin=alt.Bin(maxbins=10)),
    y="sum(solved):Q",
    color="run:N",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "mean(solved):Q", "sum(solved):Q"]
).add_params(
    selection
)
display(chart)


In [ ]:
df = results_2000_df.copy()
df = pd.merge(df, lemma_size_df, on="theorem", how="inner")
# smooth out the data
# first take mean over all points with same lemma distance
df["solved"] = df.groupby(["run", "lemma_size"])[["solved"]].transform("mean")
# use ewm to average data
df = df.sort_values("lemma_size")
df["smooth_solved"] = df.groupby("run")["solved"].transform(lambda x: x.ewm(halflife=100).mean())
#df["smooth_solved"] = df.groupby("run")["solved"].transform(lambda x: x.rolling(100).mean())
# replace with mean
df = df.groupby(["run", "lemma_size"])[["smooth_solved"]].mean().reset_index()

selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(df).mark_line().encode(
    x=alt.X("lemma_size:Q", scale=alt.Scale(type="log")),
    y="smooth_solved:Q",
    color="run:N",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N"]
).add_params(
    selection
)


In [ ]:
df = results_500_each_df.copy()
df = pd.merge(df, lemma_size_df, on="theorem", how="inner")
df = pd.merge(df, lemma_distance_df, on="theorem", how="inner")
df["lemma_distance"] = df["lemma_distance"] + 1
df["lemma_distance"] = np.log(df["lemma_distance"])
df["lemma_size"] = np.log(df["lemma_size"])
df = df[df["run"] == "k-NN"]
selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(df).mark_rect().encode(
    x=alt.X("lemma_size:Q", bin=True),
    y=alt.Y("lemma_distance:Q", bin=True),
    color="count():Q",
    #opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N"]
).add_params(
    selection
)


In [ ]:
df = results_500_each_df.copy()
df = pd.merge(df, lemma_size_df, on="theorem", how="inner")
df = pd.merge(df, lemma_distance_df, on="theorem", how="inner")
df["lemma_distance"] = df["lemma_distance"] + 1
df = df[df["solved"]]
selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(df).mark_circle().encode(
    x=alt.X("lemma_size:Q", scale=alt.Scale(type="log")),
    y=alt.Y("lemma_distance:Q", scale=alt.Scale(type="log")),
    color="run:N",
    opacity=alt.condition(selection, alt.value(.1), alt.value(0.0)),
    tooltip=["run:N"]
).add_params(
    selection
)


In [ ]:
for k, df in results_500_each_df.copy().groupby("package"):
    display(k)
    df = pd.merge(df, lemma_size_df, on="theorem", how="inner")
    df = pd.merge(df, lemma_distance_df, on="theorem", how="inner")
    df["lemma_distance"] = df["lemma_distance"] + 1
    df = df[df["solved"]]
    selection = alt.selection_point(fields=['run'], bind='legend')
    chart = alt.Chart(df).mark_circle().encode(
        x=alt.X("lemma_size:Q", scale=alt.Scale(type="quantize")),
        y=alt.Y("lemma_distance:Q", scale=alt.Scale(type="quantize")),
        color="run:N",
        opacity=alt.condition(selection, alt.value(.5), alt.value(0.0)),
        tooltip=["run:N"]
    ).add_params(
        selection
    )
    display(chart)


### Lemma distance plots for paper


In [ ]:
# for paper?
df = results_combined_2000_df.copy()
df = df[~df["package"].isin(["coq-hott.8.11", "coq-tlc.20200328"])]
df = df[df["run"].isin(paper_models_table["run"])]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])

df = pd.merge(df, lemma_distance_df, on="theorem", how="inner")
df["lemma_distance"] += 1
df["log_lemma_distance"] = np.log(df["lemma_distance"])
df1 = df[df["solved"]].copy()
df1["atype"] = "theorems solved"
chart1 = alt.Chart(df1).mark_line().transform_density(
    'log_lemma_distance',
    counts=True,
    as_=['xlog_lemma_distance', 'nsolved'],
    groupby=['run', "atype"]
).transform_calculate(
    xx="exp(datum.xlog_lemma_distance)"
).encode(
    x=alt.X("xx:Q", title="number of new dependencies (+ 1)").scale(type="log", domain=[1, 1000]),
    y=alt.Y('nsolved:Q', title="Theorems solved (smoothed)"),
    color=alt.Color("run:N", title=None).scale(domain=list(paper_models_table["model"]), range=list(paper_models_table["color"])).legend(labelLimit = 400,),
    strokeDash=alt.StrokeDash("atype:N", title=None).scale(domain=["theorems solved", "total theorem count"]),
)

df2 = df[df["run"] == "k-NN"].copy()
df2["atype"] = "total theorem count"
chart2 = alt.Chart(df2).mark_line(color="darkgray").transform_density(
    'log_lemma_distance',
    counts=True,
    as_=['xlog_lemma_distance', 'nsolved'],
    groupby=['run', "atype"]
).transform_calculate(
    xx="exp(datum.xlog_lemma_distance)"
).encode(
    x=alt.X("xx:Q", title="number of new dependencies (+ 1)").scale(type="log", domain=[1, 1000]),
    y=alt.Y('nsolved:Q', title="Theorems solved (smoothed)"),
    strokeDash=alt.StrokeDash("atype:N", title=None).scale(domain=["theorems solved", "total theorem count"]),
).properties(
    width=300
)

chart = chart1 + chart2
chart.save("paper_images/dependency_distance_solved_per_distance_plot.png", scale_factor=4)
chart

In [ ]:
# for paper?
df = results_combined_2000_df.copy()
df = pd.merge(df, lemma_distance_df, on=["theorem", "package"], how="inner")
lemma_dist_groups = [x + " new dependencies" for x in ["0", "1-10", "11-100", "101+"]]
df["lemma_dist_group"] = pd.cut(df["lemma_distance"], bins=[-1, 0.5, 10.5, 100.5, 100000], labels=lemma_dist_groups)

df = df[df["run"].isin(paper_models_table["run"])]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])

time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run", "lemma_dist_group"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
)

time_df = add_final_times(
    time_df,
    groupby=["run", "lemma_dist_group"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=10*60.0
)

time_chart1 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    #column=alt.Row("lemma_dist_group:O"),
    #color=alt.Color("run:N", title=None, scale=alt.Scale(domain=list(paper_models_table["model"]), range=list(paper_models_table["color"]))).legend(orient="top-left", fillColor="white", padding=5, offset=6, strokeColor="lightgray", labelLimit = 400,),
    color=alt.Color("run:N", title=None, scale=alt.Scale(domain=list(paper_models_table["model"]), range=list(paper_models_table["color"]))).legend(orient="none", legendX=330, legendY=300, fillColor="white", padding=5, offset=6, strokeColor="lightgray", labelLimit = 400,),

).properties(
    height=250
).facet(
    facet=alt.Facet('lemma_dist_group:N', title=None, sort=lemma_dist_groups),
    columns=2,
)

chart = time_chart1
chart.save("paper_images/dependency_distance_grid_plot.png", scale_factor=4)
chart


In [ ]:
# for paper?
runs = [
    "G2T-NoDef-Frozen",
    "G2T-Anon-Update",
    "Transformer-GPU",
    "k-NN",
]
is_online = {
    "G2T-NoDef-Frozen": "offline",
    "G2T-Anon-Update": "online",
    "Transformer-GPU": "offline",
    "k-NN": "online",
}

df = results_combined_2000_df.copy()
df = pd.merge(df, lemma_distance_df, on=["theorem", "package"], how="inner")
lemma_dist_groups = [x + " new dependencies" for x in ["0", "1-10", "11-100", "101+"]]
df["lemma_dist_group"] = pd.cut(df["lemma_distance"], bins=[-1, 0.5, 10.5, 100.5, 100000], labels=lemma_dist_groups)

paper_table_copy = paper_models_table.copy()
paper_table_copy = paper_table_copy[paper_table_copy["model"].isin(runs)]

display(paper_table_copy)
df = df[df["run"].isin(paper_table_copy["run"])]
df["run"] = df["run"].map(paper_table_copy.set_index("run")["model"])


time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run", "lemma_dist_group"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
)
display(time_df["package"].unique())
time_df = add_final_times(
    time_df,
    groupby=["run", "lemma_dist_group"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=10*60.0
)
time_df["online"] = time_df["run"].map(is_online)

time_chart1 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds", axis=alt.Axis(grid=False)),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    strokeDash=alt.StrokeDash("online:O", title=None).legend(orient="top"),
    #column=alt.Row("lemma_dist_group:O"),
    color=alt.Color("run:N", title=None, scale=alt.Scale(domain=list(paper_table_copy["model"]) + [""], range=list(paper_table_copy["color"]) + [""])).legend(orient="top", labelLimit = 400,),

).properties(
    height=100,
    width=100,
).facet(
    facet=alt.Facet('lemma_dist_group:N', title=None, sort=lemma_dist_groups),
    columns=4,
)

chart = time_chart1
chart.save("paper_images/dependency_distance_grid_plot_small.png", scale_factor=4)
chart


In [ ]:
# for paper?
df = results_combined_2000_df.copy()
df = pd.merge(df, lemma_distance_df, on=["theorem", "package"], how="inner")
lemma_dist_groups = [x + " new dependencies" for x in ["0", "1-10", "11-100", "101+"]]
df["lemma_dist_group"] = pd.cut(df["lemma_distance"], bins=[-1, 0.5, 10.5, 100.5, 100000], labels=lemma_dist_groups)

df = df[df["run"].isin(paper_models_table["run"])]
df["run"] = df["run"].map(paper_models_table.set_index("run")["model"])

time_df = aggregate_by_time_step(
    df[df["run"].map(paper_models_table.set_index("model")["steps"])],
    time_column="model_calls",
    groupby=["run", "lemma_dist_group"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
    up_to_percentile=0.5
)
def correct_model_calls(row):
    if row["run"] == "k-NN":
        return row
    else:
        row["model_calls"] -= 1
        return row 
time_df = time_df.apply(correct_model_calls, axis=1)
time_chart1 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("model_calls:Q", scale=alt.Scale(type="log"), title="model calls"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    #column=alt.Row("lemma_dist_group:O"),
    color=alt.Color("run:N", title=None, scale=alt.Scale(domain=list(paper_models_table[paper_models_table["steps"]]["model"]), range=list(paper_models_table[paper_models_table["steps"]]["color"]))).legend(orient="none", legendX=330, legendY=300, fillColor="white", padding=5, offset=6, strokeColor="lightgray", labelLimit = 400,),
).properties(
    height=250
).facet(
    facet=alt.Facet('lemma_dist_group:N', title=None, sort=lemma_dist_groups),
    columns=2,
)

chart = time_chart1
chart.save("paper_images/dependency_distance_grid_plot_model_calls.png", scale_factor=4)
chart


## More pairs of combinations

In [ ]:
knn_paper_models_table = pd.DataFrame([
    {"run": "Top 2 combined", "model": "G2T-Anon-Update + k-NN", "venn_color": "#FF5400", "color": "#e45756", "steps": True},
    {"run": "knn + G2T-Named-Update", "model": "G2T-Named-Update + k-NN", "venn_color": "lightgrey", "color": "#4c78a8", "steps": True},
    {"run": "knn + G2T-NoDef-Frozen", "model": "G2T-NoDef-Frozen + k-NN", "venn_color": "lightgrey", "color": "#f58518", "steps": True},
    {"run": "knn + CoqHammer", "model": "CoqHammer-combined + k-NN", "venn_color": "#00A61F", "color": "#54a24b", "steps": False},
    {"run": "knn + Transformer-GPU", "model": "Transformer-GPU + k-NN", "venn_color": "#F7FF00", "color": "#eeca3b", "steps": True},
    {"run": "k-NN", "model": "k-NN", "venn_color": "#7E00FF", "color": "#b279a2", "steps": True},
    
    
    #{"run": "Transformer GPU", "model": "Transformer-GPU", "venn_color": "#F7FF00", "color": "#eeca3b", "steps": True},
    #{"run": "Transformer CPU Big", "model": "Transformer-CPU", "venn_color": "lightgrey", "color": "#ff9da6", "steps": True},
    #{"run": "CoqHammer combined", "model": "CoqHammer combined", "venn_color": "#00A61F", "color": "#54a24b", "steps": False},
    #{"run": "firstorder eauto with *", "model": "firstorder", "venn_color": "lightgrey", "color": "#72b7b2", "steps": False},
])

df = results_combined_2000_df.copy()
df = df[df["run"].isin(knn_paper_models_table["run"])]
df["run"] = df["run"].map(knn_paper_models_table.set_index("run")["model"])

time_df = aggregate_by_time_step(
    df,
    time_column="seconds",
    groupby=["run"],
    packages_to_exclude=["coq-hott.8.11", "coq-tlc.20200328"],
    theorems_to_exclude=[],
)
display(time_df["package"].unique())
time_df = add_final_times(
    time_df,
    groupby=["run"], 
    monotone_cols=["cum_pct_solved"], 
    time_col="seconds", 
    max_time=10*60.0
)

time_chart1 = alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("seconds:Q", scale=alt.Scale(type="log"), title="seconds"),
    y=alt.Y("cum_pct_solved:Q", title="Cumulative Pass Rate"),
    color=alt.Color("run:N", title=None, scale=alt.Scale(domain=list(knn_paper_models_table["model"]), range=list(knn_paper_models_table["color"]))).legend(labelLimit = 400,),
)


chart = time_chart1
chart.save("paper_images/knn_combo_times.png", scale_factor=4)
chart

## SSR

In [ ]:
ssr_lemma_df = pd.read_csv(lemmas_ssr_data_path, sep="\t", names=["theorem"], dtype=str)
ssr_lemma_df["ssr_lemma"] = True
ssr_lemma_df

normal_lemma_df = pd.read_csv(lemmas_normal_data_path, sep="\t", names=["theorem"], dtype=str)
normal_lemma_df["ssr_lemma"] = False
normal_lemma_df

ssr_lemma_df = pd.concat([ssr_lemma_df, normal_lemma_df])
del normal_lemma_df
ssr_lemma_df

In [ ]:
df = results_500_each_df.copy()
df = pd.merge(df, ssr_lemma_df, on="theorem", how="inner")
df.pivot_table(values="solved", columns="run", index="ssr_lemma", aggfunc="mean")




In [ ]:
df = results_2000_df.copy()
df = pd.merge(df, ssr_lemma_df, on="theorem", how="inner")
df.pivot_table(values="solved", columns="run", index="ssr_lemma", aggfunc="mean")




In [ ]:
df = results_2000_df.copy()
df = pd.merge(df, ssr_lemma_df, on="theorem", how="inner")
df.groupby("run")["solved"].mean()


In [ ]:
df = results_2000_df.copy()
df = pd.merge(df, ssr_lemma_df, on="theorem", how="inner")
df.groupby("ssr_lemma").size()

In [ ]:
df = results_2000_df.copy()
df = pd.merge(df, ssr_lemma_df, on="theorem", how="inner")
df.pivot_table(values="solved", columns="run", index="ssr_lemma", aggfunc="mean")




In [ ]:
del ssr_lemma_df

In [ ]:
ssr_classification = {
    "coq-bits.1.1.0": True,
    "coq-qcert.2.2.0": False,
    "coq-ceres.0.4.0": False,
    "coq-corn.8.16.0": False,
    "coq-bytestring.0.9.0": False,
    "coq-hammer.1.3.2+8.11": False,
    "coq-gaia-stern.1.15": True,
    "coq-mathcomp-apery.1.0.1": True,
    "coq-tlc.20200328": False,
    "coq-iris-heap-lang.3.4.0": True,
    "coq-printf.2.0.0": False,
    "coq-smtcoq.2.0+8.11": False,
    "coq-topology.10.0.1": False,
    "coq-haskell.1.0.0": False,
    "coq-bbv.1.3": False,
    "coq-poltac.0.8.11": False,
    "coq-mathcomp-odd-order.1.14.0": True,
    "coq-hott.8.11": False,
}

df = results_2000_df.copy()
df["ssr_lemma"] = df["package"].map(ssr_classification)
df.pivot_table(values="solved", columns="run", index="ssr_lemma", aggfunc="mean")




In [ ]:
df = results_2000_df.copy()
df = pd.merge(df, new_tactic_df, on="theorem", how="inner")
df.pivot_table(values="solved", columns="run", index="new_tactics_in_original", aggfunc="mean")

In [ ]:
df = results_500_each_df.copy()
df = pd.merge(df, new_tactic_df, on="theorem", how="inner")
df.pivot_table(values="solved", columns="run", index=["package", "new_tactics_in_original"], aggfunc="sum")[["GNN Names Update new definitions", "GNN No names Update new definitions", "k-NN"]]

In [ ]:
df = results_2000_df.copy()

df["skip_axiom"] = ~df["found_proof"].isna() & (df["found_proof"].str.contains("skip_axiom") | df["found_proof"].str.contains("No_Empty_admitted"))
bad_skip_axiom_thms = df[df["skip_axiom"]]["theorem"].unique()
df = df[~df["theorem"].isin(bad_skip_axiom_thms)]

df = pd.merge(df, lemma_distance_df, on="theorem", how="inner")
df = pd.merge(df, new_tactic_df, on="theorem", how="inner")

df = df.groupby(["run", "lemma_distance", "new_tactics_in_original"])[["solved"]].mean().reset_index()
df["lemma_distance"] += 1
df["log_lemma_distance"] = np.log(df["lemma_distance"])
selection = alt.selection_point(fields=['run'], bind='legend')
chart = alt.Chart(df).mark_line().encode(
    x=alt.X("log_lemma_distance:Q",  bin=alt.Bin(maxbins=10)),
    y="mean(solved):Q",
    color="run:N",
    row="new_tactics_in_original:N",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "mean(solved):Q", "sum(solved):Q"]
).add_params(
    selection
)
display(chart)


## Compare GNN with and without names on TLC

In [ ]:
tlc_df = results_500_each_df.copy()
tlc_df = tlc_df[tlc_df["run"].isin(["GNN Names Update new definitions", "GNN No names Update new definitions"])]
tlc_df = tlc_df[tlc_df["package"].str.contains("tlc")]
tlc_df = tlc_df[tlc_df["solved"]]
solved_with_names = list(tlc_df[tlc_df["run"] == "GNN Names Update new definitions"]["theorem"].unique())
tlc_df = tlc_df[~tlc_df["theorem"].isin(solved_with_names)]
list(tlc_df["found_proof"])

In [ ]:
df = results_2000_df.copy()
df["skip_axiom"] = ~df["found_proof"].isna() & df["found_proof"].str.contains("skip_axiom")
bad_skip_axiom_thms = df[df["skip_axiom"]]["theorem"].unique()
df = df[~df["theorem"].isin(bad_skip_axiom_thms)]
df.pivot_table(values="solved", index="package", columns="run", aggfunc="sum")

In [ ]:
df = results_500_each_df.copy()
df = df[df["run"].isin(["GNN Names Update new definitions", "GNN No names Update new definitions"])]
df = df[df["package"].str.contains("coq-hott.8.11")]
df = df[df["solved"]]
solved_with_names = list(df[df["run"] == "GNN Names Update new definitions"]["theorem"].unique())
df = df[~df["theorem"].isin(solved_with_names)]
list(df["found_proof"])

In [ ]:
df = results_500_each_df.copy()
df["skip_axiom"] = ~df["found_proof"].isna() & df["found_proof"].str.contains("No_Empty_admitted")
#df["skip_axiom"] = df["skip_axiom"] | (~df["found_proof"].isna() & df["found_proof"].str.contains("No_Empty_Admitted"))
df.pivot_table(values="skip_axiom", index="package", columns="run", aggfunc="sum")

In [ ]:
df = results_500_each_df.copy()
df["skip_axiom"] = ~df["found_proof"].isna() & (df["found_proof"].str.contains("skip_axiom") | df["found_proof"].str.contains("No_Empty_admitted") | df["found_proof"].str.contains("undefined"))
bad_skip_axiom_thms = df[df["skip_axiom"]]["theorem"].unique()
df = df[~df["theorem"].isin(bad_skip_axiom_thms)]
df.pivot_table(values="solved", index="package", columns="run", aggfunc="sum")[["GNN Names Update new definitions", "GNN No names Update new definitions", "k-NN"]]

In [ ]:
df = results_500_each_df.copy()
df = df[df["run"].isin(["k-NN", "GNN No names Update new definitions"])]
df = df[df["package"].str.contains("hott")]
df = df[df["solved"]]
solved_with_names = list(df[df["run"] == "GNN No names Update new definitions"]["theorem"].unique())
df = df[~df["theorem"].isin(solved_with_names)]
list(df["found_proof"])

In [ ]:
df = results_500_each_df.copy()

df["tactics"] = df["found_proof"].str.extractall("only 1: ([\w\d]*)[^;)]*").groupby(level=0)[0].apply(set)
df["tactics"] = df["tactics"].map(lambda s: s if isinstance(s, set) else {})
tactics_used_in_gnn = [s for s in list(df[df["run"].str.contains("GNN")]["tactics"]) if isinstance(s, set)]
tactics_used_in_gnn = {t for s in tactics_used_in_gnn for t in s}
df["new_tactics"] = df["tactics"].map(lambda s: any((t not in tactics_used_in_gnn) for t in s))

#df = df[df["run"].isin(["k-NN", "GNN No names Update new definitions"])]
#solved_with_new_tactics = list(df[df["new_tactics"]]["theorem"].unique())
#df = df[df["theorem"].isin(solved_with_new_tactics)]
df.pivot_table(values="new_tactics", index="package", columns="run", aggfunc="mean")

In [ ]:
df = results_500_each_df.copy()

df["tactics"] = df["found_proof"].str.extractall("only 1: ([\w\d]*)[^;)]*").groupby(level=0)[0].apply(set)
df["tactics"] = df["tactics"].map(lambda s: s if isinstance(s, set) else {})
tactics_used_in_gnn = [s for s in list(df[df["run"].str.contains("GNN")]["tactics"]) if isinstance(s, set)]
tactics_used_in_gnn = {t for s in tactics_used_in_gnn for t in s}
df["new_tactics"] = df["tactics"].map(lambda s: any((t not in tactics_used_in_gnn) for t in s))

df = pd.merge(df, new_tactic_df, on="theorem", how="inner")

#df = df[df["run"].isin(["LSHF extreme tactic decomposition", "GNN No names Update new definitions"])]
#solved_with_new_tactics = list(df[df["new_tactics"]]["theorem"].unique())
#df = df[df["theorem"].isin(solved_with_new_tactics)]

df = df[df["solved"]]
df.pivot_table(values="new_tactics", index="new_tactics_in_original", columns="run", aggfunc="mean")

## Theorem solved when one model uses bad axioms

In [ ]:
df = results_500_each_df.copy()

df["bad_axioms"] = ~df["found_proof"].isna() & (df["found_proof"].str.contains("skip_axiom") | df["found_proof"].str.contains("No_Empty_admitted"))
bad_skip_axiom_thms = df[df["bad_axioms"]]["theorem"].unique()
df = df[df["theorem"].isin(bad_skip_axiom_thms)]
#df.groupby(["package", "run", "skip_axiom"])["solved"].count()
df.pivot_table(index=["package", "bad_axioms"], columns="run", values="solved", aggfunc="sum")



In [ ]:
#df = results_500_each_df.copy()
#
#df = df[df["package"].str.contains("mathcomp")]
#df = df[df["run"].str.contains("GNN No names Update new definitions")]
#df = df.pivot_table(index="theorem", values="found_proof", columns="run", aggfunc="first")
#df = df[df["GNN No names Update new definitions"].isna()]
#list(df["GNN No names Update new definitions - better validation"])

## Coq Hammer

In [ ]:
def get_venn_diagram_data(df):
    df = df.copy()
    df["theorem"] = pd.Categorical(df["theorem"], categories=df["theorem"].unique())
    df = df.groupby(["run", "theorem"], as_index=False).first()
    df["theorem"] = df["theorem"].astype("str")
    df["solved"] = df["solved"].fillna(False)
    
    df = df.pivot_table(index="theorem", columns="run", values="solved", aggfunc="sum").astype(bool).astype("category")
    columns = list(df.columns)
    df = df.groupby(columns).size().reset_index(name="size").set_index(columns)
    output_json = {
        "class_labels": columns,
        "data": sorted([
            {"classes": [i+1 for i, b in enumerate(ix) if b], "size": int(size)}
            for ix, size in df.iterrows()
        ], key=lambda x: (len(x["classes"]), x["classes"]))
    }
    return df.sort_values("size", ascending=False), output_json

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer")]
df, output_json = get_venn_diagram_data(df)
#with open("venn_diagram_coqhammer.json", "w") as f: json.dump(output_json, f)
df

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("GNN")]
df, output_json = get_venn_diagram_data(df)
df

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("GNN Names") | df["run"].str.contains("GNN names")]
df, output_json = get_venn_diagram_data(df)
#with open("venn_diagram_gnn_names.json", "w") as f: json.dump(output_json, f)
df

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("GNN No Names") | df["run"].str.contains("GNN No names")]
df, output_json = get_venn_diagram_data(df)
#with open("venn_diagram_gnn_no_names.json", "w") as f: json.dump(output_json, f)
df

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].isin(["GNN No names Update new definitions", "GNN Names Update new definitions", "GNN No Definitions Update no definitions"])]
df, output_json = get_venn_diagram_data(df)
#with open("venn_diagram_gnn_best.json", "w") as f: json.dump(output_json, f)
df

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].isin(["GNN No names Update new definitions", "k-NN", "Transformer GPU", "firstorder eauto with *"])]
df, output_json = get_venn_diagram_data(df)
#with open("venn_diagram_firstorder_vs_our_models.json", "w") as f: json.dump(output_json, f)
df

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].isin(["LSHF", "LSHF extreme tactic decomposition", "k-NN"])]
df, output_json = get_venn_diagram_data(df)
#with open("venn_diagram_knn_models.json", "w") as f: json.dump(output_json, f)
df

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].isin(["CoqHammer 'best' tactic", "CoqHammer 'sauto' tactic", "firstorder eauto with *", "CoqHammer Vampire"])]
df, output_json = get_venn_diagram_data(df)
df

## Best combination of models in 10 minutes

Have monotonic set functions f_n(t).  Want to find the optimal combination $t_0 + ... + t_n$ such that max: union f_n(t_n) and sum t_n = 10min.

In [ ]:
results_2000_time_df = results_2000_df.copy()
results_2000_time_df["time"] = np.where(results_2000_time_df["solved"], results_2000_time_df["time"], np.inf)
results_2000_time_df = results_2000_time_df.pivot_table(index="theorem", columns="run", values="time")
results_2000_time_df = results_2000_time_df.fillna(np.inf)
results_2000_time_df

In [ ]:
(results_2000_time_df["CoqHammer 'best' tactic"] <= 600.0).sum()

In [ ]:
df = results_2000_df.copy()
df = df[df["run"] == "CoqHammer 'best' tactic"]
(df["solved"] & (df["time"] <= 600.0)).sum()

In [ ]:
coq_hammer_vampire_combo = {"CoqHammer Vampire": .5, "CoqHammer 'best' tactic": .5}
get_combo_size(results_2000_time_df, coq_hammer_vampire_combo, 10*60.0)

In [ ]:
coq_hammer_vampire_combo = {"CoqHammer Vampire": (10.0*60.0 - 15.0) / (10.0*60.0), "CoqHammer 'best' tactic": 15.0 / (10.0*60.0)}
get_combo_size(results_2000_time_df, coq_hammer_vampire_combo, 10.0*60.0)

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer") & ~df["run"].str.contains("sauto")]
coq_hammers = df["run"].unique()
coq_hammers_combo = {ch: 1/len(coq_hammers) for ch in coq_hammers}
get_combo_size(results_2000_time_df, coq_hammers_combo, 10.0*60.0)

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer") & ~df["run"].str.contains("sauto")]
coq_hammers = df["run"].unique()
coq_hammers_combo = {ch: 1/(len(coq_hammers)-1) * (10*60.0 - 15)/(10.0*60.0) for ch in coq_hammers if "best" not in coq_hammers}
coq_hammers_combo["CoqHammer 'best' tactic"] = 15.0 / (10.0*60.0)
get_combo_size(results_2000_time_df, coq_hammers_combo, 10.0*60.0)

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer")]
coq_hammers = df["run"].unique()
coq_hammers_combo = {ch: 1/len(coq_hammers) for ch in coq_hammers}
get_combo_size(results_2000_time_df, coq_hammers_combo, 10.0*60.0)

In [ ]:
df = results_2000_df.copy()
runs = df["run"].unique()
runs_combo = {ch: 1/len(runs) for ch in runs}
get_combo_size(results_2000_time_df, runs_combo, 10.0*60.0)

In [ ]:
df = results_2000_df.copy()
df = df[~df["run"].str.contains("Transformer")]
runs = df["run"].unique()
runs_combo = {ch: 1/len(runs) for ch in runs}
get_combo_size(results_2000_time_df, runs_combo, 10.0*60.0)

In [ ]:
import random

def get_best_combo(results_time_df, runs, time_limit):
    results_time_df = results_time_df.copy()[runs]
    combos = []
    sizes = []
    times = []
    best_size = 0
    best_time = np.inf
    best_score = 0
    best_combo = None
    for i in range(10):
        combo = {r: random.random() for r in runs}
        combo_sum = sum(combo.values())
        combo = {r: t/combo_sum for r, t in combo.items()}
        combos.append(combo)
        sizes.append(0)
        times.append(np.inf)

    for i in range(1000):
        temp = (1000 - i) / 1000
        print(i, best_size, best_time, best_score, best_combo)
        new_combos = []
        new_sizes = []
        new_times = []
        for combo in combos:
            combo_delta = {r: random.random() * temp for r in runs}
            combo = {r: combo[r] + combo_delta[r] for r in combo}
            combo_sum = sum(combo.values())
            combo = {r: t/combo_sum for r, t in combo.items()}
            new_combos.append(combo)
            combo = {r: combo[r] for r in combo}
            new_sizes.append(get_combo_size(results_time_df, combo, time_limit))
            new_times.append(get_combo_time(results_time_df, combo, time_limit))
        combos = combos + new_combos
        sizes = sizes + new_sizes
        times = times + new_times
        combos_scores_time = zip(combos, sizes, times)
        combos_scores_time = sorted(combos_scores_time, key=lambda cst: 10*time_limit*(2000-cst[1]) + cst[2])
        combos_scores_time = combos_scores_time[:10]
        combos = [c for c, s, t in combos_scores_time]
        sizes = [s for c, s, t in combos_scores_time]
        times = [t for c, s, t in combos_scores_time]
        best_combo = combos[0]
        best_size = sizes[0]
        best_time = times[0]
        best_score = time_limit*2000 - time_limit*(2000-best_size) - best_time

In [ ]:
import random

def get_best_combo_sa(results_time_df, runs, time_limit):
    results_time_df = results_time_df.copy()[runs]
    best_size = 0
    best_time = np.inf
    best_score = -np.inf
    best_combo = None
    score = -np.inf
    combo = {r: 0.0 for r in runs}

    max_steps = 10000
    for i in range(max_steps):
        temp = (max_steps - i) / max_steps
        if i % 100 == 0:
            print(i, best_size, best_time, best_score, best_combo)
            print(i,)
        combo_delta = {r: random.random() * temp for r in runs}
        new_combo = {r: combo[r] + combo_delta[r] for r in combo}
        new_combo_sum = sum(new_combo.values())
        new_combo = {r: t/new_combo_sum for r, t in new_combo.items()}
        new_size = get_combo_size(results_time_df, new_combo, time_limit)
        new_time = get_combo_time(results_time_df, new_combo, time_limit)
        new_score = time_limit*2000 - time_limit*(2000-new_size) - new_time
        if i % 100 == 0:
            print(i, new_size, new_time, new_score, new_combo)
            
        if new_score > best_score:
            best_time = new_time
            best_size = new_size
            best_score = new_score
            best_combo = new_combo
        
        if new_score >= score:
            score = new_score
            combo = new_combo
        else:
            ratio = np.exp(new_score - score)
            if ratio > 0: #random.random():
                score = new_score
                combo = new_combo

In [ ]:
get_best_combo(results_2000_time_df, ["CoqHammer Vampire", "CoqHammer 'best' tactic"], 10.0*60.0)

In [ ]:
get_best_combo_sa(results_2000_time_df, ["CoqHammer Vampire", "CoqHammer 'best' tactic"], 10.0*60.0)

In [ ]:
get_best_combo(results_2000_time_df, coq_hammers, 10*60.0)

In [ ]:
get_best_combo(results_2000_time_df, list(results_2000_time_df.columns), 10*60.0)

In [ ]:
get_best_combo(results_2000_time_df, [r for r in list(results_2000_time_df.columns) if not r.startswith("Transformer")], 10.0*60.0)

In [ ]:
get_best_combo(results_2000_time_df, ["GNN No names Update no definitions", "k-NN", "Transformer GPU", "CoqHammer 'best' tactic"], 10.0*60.0)

In [ ]:
get_best_combo(results_2000_time_df, ["GNN No names Update no definitions", "k-NN"], 10.0*60.0)

## Combined hammer for venn diagram

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer")]
coq_hammers = df["run"].unique()
coq_hammers_combo = {ch: 1/len(coq_hammers) for ch in coq_hammers}
combined_hammer_df = get_combo(results_2000_time_df, coq_hammers_combo, 10.0*60.0).copy()
combined_hammer_df = combined_hammer_df.reset_index()
combined_hammer_df["solved"] = True
combined_hammer_df["run"] = "CoqHammer combined"
combined_hammer_df

In [ ]:
# all packages
df = results_combined_2000_df.copy()
df = df[df["run"].isin(["GNN No names Update new definitions", "k-NN", "Transformer GPU", "CoqHammer combined"])]
df["run"] = df["run"].map({
    "GNN No names Update new definitions": "G2T-Anon-Update",
    "k-NN": "k-NN",
    "Transformer GPU": "Transformer GPU",
    "CoqHammer combined": "CoqHammer combined",
})
df, output_json = get_venn_diagram_data(df)
#with open("venn_diagram_coqhammer_vs_our_models2.json", "w") as f: json.dump(output_json, f)
df / df["size"].sum()

In [ ]:
# all packages
df = results_combined_2000_df.copy()
df = df[~df["package"].isin(["coq-hott.8.11", "coq-tlc.20200328"])]
df = df[df["run"].isin(["GNN No names Update new definitions", "k-NN", "Transformer GPU", "CoqHammer combined"])]
df["run"] = df["run"].map({
    "GNN No names Update new definitions": "G2T-Anon-Update",
    "k-NN": "k-NN",
    "Transformer GPU": "Transformer GPU",
    "CoqHammer combined": "CoqHammer combined",
})
df, output_json = get_venn_diagram_data(df)
#with open("venn_diagram_coqhammer_vs_our_models3.json", "w") as f: json.dump(output_json, f)
df / df["size"].sum()

## Redo plots with coq hammer aggregate

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer") & ~df["run"].str.contains("sauto")]
df = df.drop(columns="time")
coq_hammers = df["run"].unique()
coq_hammers_combo = {ch: 1/len(coq_hammers) for ch in coq_hammers}
combined_hammer_df = get_combo(results_2000_time_df, coq_hammers_combo, 10*60.0).copy()
combined_hammer_df = combined_hammer_df.reset_index().set_index(["theorem", "run"])
combined_hammer_df = combined_hammer_df.join(df.set_index(["theorem", "run"]))
# add back in all unsolved theorems
df = df[df["run"].str.contains("best")]
df = df.set_index(["package_theorem", "theorem", "package"])[[]]
combined_hammer_df = df.join(combined_hammer_df.reset_index().set_index(["package_theorem", "theorem", "package"])).reset_index()
combined_hammer_df["run"] = "CoqHammer combined"
combined_hammer_df["solved"] = combined_hammer_df["solved"].fillna(False)
combined_hammer_df["error"] = combined_hammer_df["error"].fillna(False)
combined_hammer_df["omitted"] = combined_hammer_df["omitted"].fillna(False)
combined_hammer_df["steps"] = combined_hammer_df["steps"].fillna(0)
combined_hammer_df["messages"] = combined_hammer_df["messages"].fillna(0)
combined_hammer_df


In [ ]:
# solved per unit of time
df = results_2000_df.copy()
df = pd.concat([df, combined_hammer_df])
df = df[df["package"] != "coq-hott.8.11"]
#df = df[df["package"] != "coq-tlc.20200328"]
#df = df[~df.groupby("theorem")["error"].transform("max")]

#df["skip_axiom"] = ~df["found_proof"].isna() & (df["found_proof"].str.contains("skip_axiom") | df["found_proof"].str.contains("No_Empty_admitted"))
#bad_skip_axiom_thms = df[df["skip_axiom"]]["theorem"].unique()
#df = df[~df["theorem"].isin(bad_skip_axiom_thms)]


df["package_thm"] = df["package"] + df["theorem"]

df["run_total"] = df.groupby("run")["solved"].transform("count")

# weight
#df = df.merge(right=package_sizes[["package", "total"]], how="left", on="package")
#df["weight"] = df["total"]
#df["_solved"] = df["solved"]
#df["solved"] = df["solved"].astype("int") * df["weight"]
#df["run_total"] = df.groupby("run")["weight"].transform("sum")

df["pct_solved"] = df["solved"] / df["run_total"]
display(df[df["run"].str.contains("combined")])
df = df[~df["error"]]
df["steps"] = df["steps"].astype("int")
df["messages"] = df["messages"].astype("int")
df["time_ix"] = df.sort_values(["time"]).groupby(["run"])["time"].cumcount()
df["steps_ix"] = df.sort_values(["steps"]).groupby(["run"])["steps"].cumcount()
df["messages_ix"] = df.sort_values(["messages"]).groupby(["run"])["messages"].cumcount()
display(df[df["run"].str.contains("combined")]["solved"])
df = df[df["solved"]]

# sort by time
time_df = df.copy().sort_values(["time"])
time_df["cum_solved"] = time_df.groupby(["run"])["solved"].cumsum()
time_df["cum_pct_solved"] = time_df["cum_solved"] / time_df["run_total"]
time_df["cum_steps"] = time_df.groupby(["run"])["steps"].cumsum()
time_df["cum_time"] = time_df.groupby(["run"])["time"].cumsum()
time_df["steps_per_second"] = time_df["cum_steps"] / time_df["cum_time"]
time_df

# sort by tactics executed
step_df = df.copy().sort_values(["steps"])
step_df["cum_solved"] = step_df.groupby(["run"])["solved"].cumsum()
step_df["remaining"] = step_df["run_total"] - step_df.groupby(["run"])["solved"].cumcount()
step_df["cum_pct_solved"] = step_df["cum_solved"] / time_df["run_total"]  # (step_df["cum_solved"] + (step_df["run_total"] - step_df["steps_ix"]))
step_df["cum_steps"] = step_df.groupby(["run"])["steps"].cumsum()
step_df["cum_time"] = step_df.groupby(["run"])["time"].cumsum()
step_df["steps_per_second"] = step_df["cum_steps"] / step_df["cum_time"]
step_df = step_df[step_df["steps_ix"] <= step_df["run_total"] * .5]
step_df

# sort by messages
step2_df = df.copy().sort_values(["messages"])
step2_df["cum_solved"] = step2_df.groupby(["run"])["solved"].cumsum()
step2_df["cum_pct_solved"] = step2_df["cum_solved"] / time_df["run_total"]  # (step2_df["cum_solved"] + (step2_df["run_total"] - step2_df["messages_ix"]))
step2_df["cum_messages"] = step2_df.groupby(["run"])["messages"].cumsum()
step2_df["cum_time"] = step2_df.groupby(["run"])["time"].cumsum()
step2_df["messages_per_second"] = step2_df["cum_messages"] / step2_df["cum_time"]
step2_df = step2_df[step2_df["messages_ix"] <= step2_df["run_total"] * .5]
step2_df

time_df


In [ ]:
selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("time:Q"),
    y=alt.Y("cum_pct_solved:Q"),
    color="run:N",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "cum_pct_solved:Q", "time:Q"]
).add_params(
    selection
)

In [ ]:
selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(time_df).mark_line(clip=True).encode(
    x=alt.X("time:Q", scale=alt.Scale(type="log")),
    y=alt.Y("cum_pct_solved:Q"),
    color="run",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "cum_pct_solved:Q", "cum_solved:Q", "time:Q"]
).add_params(
    selection
)

In [ ]:
selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(step_df).mark_line().encode(
    x=alt.X("steps:Q", scale=alt.Scale(type="log")),
    y="cum_pct_solved:Q",
    color="run",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "cum_pct_solved:Q", "steps:Q"]
).add_params(
    selection
)

In [ ]:
selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(step2_df).mark_line().encode(
    x=alt.X("messages:Q", scale=alt.Scale(type="log")),
    y="cum_pct_solved:Q",
    color="run",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "cum_pct_solved:Q", "messages:Q"]
).add_params(
    selection
)

In [ ]:
alt.Chart(step_df).mark_line().encode(
    x=alt.X("time:Q", scale=alt.Scale(type="log")),
    y=alt.Y("steps:Q", scale=alt.Scale(type="log")),
    color="run",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "steps:Q", "time:Q"]
).add_params(
    selection
)

In [ ]:
selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(step_df).mark_line().encode(
    x=alt.X("time:Q", scale=alt.Scale(type="log")),
    y=alt.Y("steps_per_second:Q", scale=alt.Scale(type="log")),
    color="run",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "steps_per_second:Q", "time:Q"]
).add_params(
    selection
)

In [ ]:
selection = alt.selection_point(fields=['run'], bind='legend')
alt.Chart(step2_df).mark_line().encode(
    x=alt.X("time:Q", scale=alt.Scale(type="log")),
    y=alt.Y("messages_per_second:Q", scale=alt.Scale(type="log")),
    color="run",
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=["run:N", "messages_per_second:Q", "time:Q"]
).add_params(
    selection
)

## Largest Objects in Memory

In [ ]:
import sys

# These are the usual ipython objects, including this one you are creating
ipython_vars = ['In', 'Out', 'exit', 'quit', 'get_ipython', 'ipython_vars']

# Get a sorted list of the objects and their sizes
sorted([(x, sys.getsizeof(globals().get(x))) for x in dir() if not x.startswith('_') and x not in sys.modules and x not in ipython_vars], key=lambda x: x[1], reverse=True)

## Investigating coq-ceres

In [ ]:
df = results_500_each_df.copy()
df = df[df["package"].str.contains("ceres")]
df = df[df["run"].str.contains("k-NN")]
df